
# Stream 3: Second Retailer Validation

**Project:** Stockout-Aware Demand Forecasting

## Research Objective

We investigate whether correcting recorded sales for stockout-related missed demand improves forecasting performance on a second grocery retailer.

We aim to reproduce Stream 2's core forecasting methodology on an independent dataset to evaluate generalizability.


The Rohlik dataset provides daily sales already adjusted for partial product availability using the retailer's internal methodology.

Therefore, we do not apply an additional sales/availability correction.

Instead, we design a semi-synthetic stockout experiment:

1. Treat Rohlik's adjusted sales as a proxy for potential demand.
2. Construct synthetic availability-censored sales by multiplying adjusted sales by the recorded availability fraction.
3. Compare forecasting models trained on synthetic censored sales versus models trained on adjusted sales.
4. Use consistent chronological splits, forecast horizons, and evaluation metrics.
5. Evaluate performance against the adjusted-sales demand proxy, including forecast bias.
6. Discuss assumptions and limitations of the synthetic censoring process.

This experiment tests the effect of stockout-related sales censoring on forecasting performance. It does not directly validate the team's original missed-demand recovery model on independent, uncorrected sales.



In [139]:

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Final Project")
NOTEBOOK_DIR = PROJECT_DIR / "Notebooks"
DATA_DIR = PROJECT_DIR / "Stockout Project Data"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = DATA_DIR / "figures"

print("PROJECT:", PROJECT_DIR.exists())
print("NOTEBOOKS:", NOTEBOOK_DIR.exists())
print("DATA:", DATA_DIR.exists())
print("PROCESSED:", PROCESSED_DIR.exists())
print("FIGURES:", FIGURES_DIR.exists())

print("\nSTREAM 2 INPUT FILES:")
for filename in [
    "daily_model.parquet",
    "daily_demand_v1.parquet",
    "sample_ids.csv"
]:
    path = PROCESSED_DIR / filename
    print(filename, ":", path.exists())

print("\nSTREAM 2 NOTEBOOK:")
print(
    "stream2_forecasting.ipynb:",
    (NOTEBOOK_DIR / "stream2_forecasting.ipynb").exists()
)


PROJECT: True
NOTEBOOKS: True
DATA: True
PROCESSED: True
FIGURES: True

STREAM 2 INPUT FILES:
daily_model.parquet : True
daily_demand_v1.parquet : True
sample_ids.csv : True

STREAM 2 NOTEBOOK:
stream2_forecasting.ipynb: True



## Step 3: Shared GitHub Repository and Code Verification

We use the team's existing GitHub repository to maintain consistency with the other workstreams.

Before implementing Stream 3, we inspect the shared data-loading and evaluation functions and identify the forecasting methodology used in Stream 2.


In [140]:

from pathlib import Path
import subprocess

REPO_URL = "https://github.com/bulut19/stockout-aware-forecasting.git"
REPO_DIR = Path("/content/stockout-aware-forecasting")

if REPO_DIR.exists():
    print("Repository already exists.")
else:
    result = subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        capture_output=True,
        text=True
    )
    print("Clone return code:", result.returncode)
    if result.returncode != 0:
        print(result.stderr[-1000:])

if REPO_DIR.exists():
    print("\nREPOSITORY FILES:")
    for path in sorted(REPO_DIR.iterdir()):
        print("-", path.name)

    print("\nSHARED SOURCE FILES:")
    for filename in ["src/load_data.py", "src/metrics.py"]:
        path = REPO_DIR / filename
        print(filename, ":", path.exists())

    print("\nCURRENT BRANCH:")
    subprocess.run(["git", "-C", str(REPO_DIR), "branch", "--show-current"])


Repository already exists.

REPOSITORY FILES:
- .git
- .gitignore
- README.md
- figures
- notebooks
- requirements.txt
- results
- src

SHARED SOURCE FILES:
src/load_data.py : True
src/metrics.py : True

CURRENT BRANCH:



## Step 4: Shared Code Inspection

We inspect the existing data-loading and evaluation utilities before implementing Stream 3.

Our goal is to reuse the team's shared functions and follow the same forecasting and evaluation conventions, while adapting the pipeline to a second retailer.


In [141]:

from pathlib import Path

for filename in ["src/load_data.py", "src/metrics.py"]:
    path = REPO_DIR / filename

    print("\n" + "=" * 60)
    print(filename)
    print("=" * 60)

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("FILE NOT FOUND")



src/load_data.py
"""Shared data loader for FreshRetailNet-50K.

Usage in Colab (after the setup cell):
    from src.load_data import download_raw, build_daily, load_daily, load_hourly
    download_raw()          # once: downloads the raw files to the shared Drive folder
    build_daily()           # once: builds and saves the daily table
    daily = load_daily()    # every time after that
    hourly = load_hourly(series_ids=[...])  # hourly data for chosen series only
"""
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

REPO_ID = "Dingdong-Inc/FreshRetailNet-50K"
# The data folder sits in a different place for each person, so use the first one that exists
_CANDIDATES = [
    "/content/drive/MyDrive/MIS 382 Deep Learning and ML/Final Project/Stockout Project Data",  # Melisa
    "/content/drive/MyDrive/Final Project/Stockout Project Data",  # teammates with a shortcut to Final Project
    "/content/drive/MyDrive/Stockout Project Data",  # t

In [142]:

for folder_name in ["results", "figures", "notebooks"]:
    folder = REPO_DIR / folder_name

    print(f"\n--- {folder_name.upper()} ---")

    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                print(path.relative_to(REPO_DIR))
    else:
        print("Folder not found")



--- RESULTS ---
results/figures/README.md
results/stream3_second_retailer_findings.md
results/tables/README.md
results/tables/results_stream3_by_stockout_v1.csv
results/tables/results_stream3_by_warehouse_v1.csv
results/tables/results_stream3_v1.csv
results/tables/results_stream5_by_week_v1.csv
results/tables/results_stream5_percentile_curve_v1.csv
results/tables/results_stream5_sensitivity_v1.csv
results/tables/results_stream5_stability_v1.csv
results/tables/results_stream5_stream2_files_v1.csv
results/tables/results_stream5_v1.csv

--- FIGURES ---
figures/stream2_bias.png
figures/stream2_example_product.png
figures/stream2_tft_example.png
figures/stream3_actual_vs_forecast.png
figures/stream3_stockout_severity.png
figures/stream3_test_bias_comparison.png
figures/stream3_test_wape_comparison.png
figures/stream3_warehouse_improvement.png
figures/stream5_cost_by_model.png
figures/stream5_cost_by_percentile.png
figures/stream5_gain_sensitivity.png

--- NOTEBOOKS ---
notebooks/README.md



## Step 5: Second-Retailer Dataset Discovery

We inspect the existing shared project folders for a second-retailer dataset.

Before preprocessing or forecasting, we will verify the dataset's source, columns, sales quantities, product identifiers, availability indicators, and prices.

We will only estimate missed demand when the available data supports a defensible stockout correction.


In [143]:

from pathlib import Path

print("SEARCHING PROJECT DATA FILES\n")

keywords = ("rohlik", "retailer", "inventory", "stockout", "availability")

matches = []

for path in DATA_DIR.rglob("*"):
    if path.is_file() and any(
        keyword in path.name.lower()
        for keyword in keywords
    ):
        matches.append(path)

if matches:
    for path in sorted(matches):
        print(path.relative_to(DATA_DIR))
else:
    print("No second-retailer files found by filename.")

print("\nTOP-LEVEL DATA FOLDERS:")
for path in sorted(DATA_DIR.iterdir()):
    print("-", path.name)


SEARCHING PROJECT DATA FILES

figures/5_2_stockout_hours.png
figures/5_4_stockout_start.png
figures/5_5_demand_before_stockout.png
figures/stream3_stockout_severity.png
processed/results_stream3_by_stockout_v1.csv
processed/stream3_rohlik_daily_v1.parquet
processed/stream3_rohlik_enriched_v1.parquet
processed/stream3_rohlik_sample_ids_v1.csv
raw/rohlik/inventory.csv

TOP-LEVEL DATA FOLDERS:
- figures
- processed
- raw
- tft_checkpoints


In [144]:

#!pip -q install kagglehub


In [145]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

ROHLIK_DIR = Path(
    "/content/drive/MyDrive/Final Project/Stockout Project Data/raw/rohlik"
)

print("Dataset folder exists:", ROHLIK_DIR.exists())
print("Files:", [p.name for p in ROHLIK_DIR.iterdir()])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset folder exists: True
Files: ['calendar.csv', 'inventory.csv', 'sales_test.csv', 'sales_train.csv', 'solution.csv', 'test_weights.csv', '.complete']


In [146]:
from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount('/content/drive')

ROHLIK_DIR = Path(
    '/content/drive/MyDrive/Final Project/Stockout Project Data/raw/rohlik'
)

print("Folder exists:", ROHLIK_DIR.exists())

if ROHLIK_DIR.exists():
    print("Files:", [p.name for p in ROHLIK_DIR.iterdir()])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder exists: True
Files: ['calendar.csv', 'inventory.csv', 'sales_test.csv', 'sales_train.csv', 'solution.csv', 'test_weights.csv', '.complete']


In [147]:
sales = pd.read_csv(ROHLIK_DIR / 'sales_train.csv')
inventory = pd.read_csv(ROHLIK_DIR / 'inventory.csv')
calendar = pd.read_csv(ROHLIK_DIR / 'calendar.csv')

print(sales.shape)
print(sales.head())

(4007419, 14)
   unique_id        date   warehouse  total_orders  sales  sell_price_main  \
0       4845  2024-03-10  Budapest_1        6436.0  16.34           646.26   
1       4845  2021-05-25  Budapest_1        4663.0  12.63           455.96   
2       4845  2021-12-20  Budapest_1        6507.0  34.55           455.96   
3       4845  2023-04-29  Budapest_1        5463.0  34.52           646.26   
4       4845  2022-04-01  Budapest_1        5997.0  35.92           486.41   

   availability  type_0_discount  type_1_discount  type_2_discount  \
0          1.00          0.00000              0.0              0.0   
1          1.00          0.00000              0.0              0.0   
2          1.00          0.00000              0.0              0.0   
3          0.96          0.20024              0.0              0.0   
4          1.00          0.00000              0.0              0.0   

   type_3_discount  type_4_discount  type_5_discount  type_6_discount  
0              0.0      


## Step 7: Rohlik Dataset Inspection

We inspect the Rohlik dataset to understand its structure, identify the forecasting target, and determine whether it contains reliable inventory or product availability information.

We also examine the available pricing and calendar features before adapting Stream 2's forecasting pipeline.

No demand correction or model training is performed during this step.


In [148]:

import pandas as pd

files_to_inspect = [
    "sales_train.csv",
    "inventory.csv",
    "calendar.csv",
    "sales_test.csv",
    "solution.csv",
    "test_weights.csv"
]

for filename in files_to_inspect:
    path = ROHLIK_DIR / filename

    print("\n" + "=" * 65)
    print("FILE:", filename)
    print("=" * 65)

    if not path.exists():
        print("File not found")
        continue

    sample = pd.read_csv(path, nrows=5)

    print("COLUMNS:")
    print(sample.columns.tolist())

    print("\nFIRST 3 ROWS:")
    print(sample.head(3).to_string(index=False))



FILE: sales_train.csv
COLUMNS:
['unique_id', 'date', 'warehouse', 'total_orders', 'sales', 'sell_price_main', 'availability', 'type_0_discount', 'type_1_discount', 'type_2_discount', 'type_3_discount', 'type_4_discount', 'type_5_discount', 'type_6_discount']

FIRST 3 ROWS:
 unique_id       date  warehouse  total_orders  sales  sell_price_main  availability  type_0_discount  type_1_discount  type_2_discount  type_3_discount  type_4_discount  type_5_discount  type_6_discount
      4845 2024-03-10 Budapest_1        6436.0  16.34           646.26           1.0              0.0              0.0              0.0              0.0          0.15312              0.0              0.0
      4845 2021-05-25 Budapest_1        4663.0  12.63           455.96           1.0              0.0              0.0              0.0              0.0          0.15025              0.0              0.0
      4845 2021-12-20 Budapest_1        6507.0  34.55           455.96           1.0              0.0            


## Step 8: Availability and Data Quality Analysis

We examine the distribution of product availability, missing values, dates, and product-level observations in the Rohlik dataset.

This step determines whether the availability variable can support a reasonable estimate of missed demand.

We do not assume that zero sales indicate a stockout, and we do not apply demand correction until the availability definition has been verified.


In [149]:

import pandas as pd

train_path = ROHLIK_DIR / "sales_train.csv"

cols = [
    "unique_id", "date", "warehouse",
    "sales", "availability", "sell_price_main"
]

df = pd.read_csv(
    train_path,
    usecols=cols,
    parse_dates=["date"]
)

print("DATASET SHAPE:", df.shape)
print("UNIQUE PRODUCTS:", df["unique_id"].nunique())
print("UNIQUE WAREHOUSES:", df["warehouse"].nunique())
print("DATE RANGE:", df["date"].min(), "to", df["date"].max())

print("\nMISSING VALUES (%):")
print((df.isna().mean() * 100).round(2))

print("\nAVAILABILITY STATISTICS:")
print(df["availability"].describe())

print("\nAVAILABILITY VALUES (first 20):")
print(sorted(df["availability"].dropna().unique())[:20])

print("\nAVAILABILITY DISTRIBUTION:")
print("Fully available (%):",
      round((df["availability"] == 1).mean() * 100, 2))
print("Partially available (%):",
      round(((df["availability"] > 0) &
             (df["availability"] < 1)).mean() * 100, 2))
print("Zero availability (%):",
      round((df["availability"] == 0).mean() * 100, 2))

print("\nSALES STATISTICS:")
print(df["sales"].describe())

print("\nDUPLICATE PRODUCT-DATE ROWS:",
      df.duplicated(["unique_id", "date"]).sum())


DATASET SHAPE: (4007419, 6)
UNIQUE PRODUCTS: 5390
UNIQUE WAREHOUSES: 7
DATE RANGE: 2020-08-01 00:00:00 to 2024-06-02 00:00:00

MISSING VALUES (%):
unique_id          0.0
date               0.0
warehouse          0.0
sales              0.0
sell_price_main    0.0
availability       0.0
dtype: float64

AVAILABILITY STATISTICS:
count    4.007419e+06
mean     9.253107e-01
std      1.796205e-01
min      1.000000e-02
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: availability, dtype: float64

AVAILABILITY VALUES (first 20):
[np.float64(0.01), np.float64(0.02), np.float64(0.03), np.float64(0.04), np.float64(0.05), np.float64(0.06), np.float64(0.07), np.float64(0.08), np.float64(0.09), np.float64(0.1), np.float64(0.11), np.float64(0.12), np.float64(0.13), np.float64(0.14), np.float64(0.15), np.float64(0.16), np.float64(0.17), np.float64(0.18), np.float64(0.19), np.float64(0.2)]

AVAILABILITY DISTRIBUTION:
Fully available (%): 75.74
Partially availa


### Initial Rohlik Data Quality Findings

The Rohlik training dataset contains 4,007,419 records across 5,390 product series and 7 warehouses, spanning August 2020 to June 2024.

Approximately 75.74% of observations have full availability, while 24.26% have partial availability. No observations have zero availability, and there are no duplicate product-date records.

The data includes recorded sales, product availability, and selling prices. This makes it a promising second-retailer dataset for investigating stockout-aware demand forecasting.

We also identify 52 missing sales observations that require investigation before preprocessing.

Unlike FreshRetailNet, which provides hourly stockout flags, Rohlik provides a daily availability measure. Therefore, the stockout measures are not directly comparable, and we must verify the precise meaning of Rohlik's availability field before correcting demand.


In [150]:

import pandas as pd
import numpy as np

print("MISSING SALES ROWS:")
missing = df[df["sales"].isna()]
print(missing.head(10).to_string(index=False))
print("Total missing sales:", len(missing))

print("\nMISSING SALES BY WAREHOUSE:")
print(missing["warehouse"].value_counts())

print("\nAVAILABILITY GROUPS:")

bins = [0, 0.25, 0.5, 0.75, 0.99, 1.0]
labels = [
    "1–25%",
    "25–50%",
    "50–75%",
    "75–99%",
    "100%"
]

df["availability_group"] = pd.cut(
    df["availability"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

availability_summary = (
    df.groupby("availability_group", observed=True)
      .agg(
          observations=("sales", "size"),
          average_sales=("sales", "mean"),
          median_sales=("sales", "median"),
          average_price=("sell_price_main", "mean")
      )
)

availability_summary["percent_of_data"] = (
    availability_summary["observations"] / len(df) * 100
)

print(availability_summary.round(2).to_string())

print("\nZERO-SALES OBSERVATIONS:")
print("Zero sales:", (df["sales"] == 0).sum())
print(
    "Zero sales with partial availability:",
    ((df["sales"] == 0) & (df["availability"] < 1)).sum()
)


MISSING SALES ROWS:
 unique_id       date   warehouse  sales  sell_price_main  availability
       885 2021-05-23    Munich_1    NaN             4.02           1.0
       885 2021-05-21    Munich_1    NaN             4.02           1.0
       885 2021-05-24    Munich_1    NaN             4.02           1.0
       885 2021-05-22    Munich_1    NaN             4.02           1.0
      1237 2021-12-10 Frankfurt_1    NaN             3.84           0.3
      1237 2021-12-09 Frankfurt_1    NaN             3.84           1.0
       725 2021-06-27    Munich_1    NaN             2.82           1.0
       725 2021-06-26    Munich_1    NaN             2.82           1.0
      3778 2021-05-29    Munich_1    NaN             2.75           1.0
      3778 2021-05-23    Munich_1    NaN             2.75           1.0
Total missing sales: 52

MISSING SALES BY WAREHOUSE:
warehouse
Munich_1       46
Frankfurt_1     6
Name: count, dtype: int64

AVAILABILITY GROUPS:
                    observations  average


### Data Quality and Availability Findings

The Rohlik dataset contains 52 missing sales values out of 4,007,419 observations. Of these, 46 occur in the Munich_1 warehouse and 6 in Frankfurt_1.

We also identify 49,207 zero-sales observations, including 27,333 observations where product availability is below 100%.

Approximately 24.26% of observations have partial availability, suggesting that product unavailability may affect a substantial portion of recorded sales.

However, average sales do not decrease monotonically with availability. This may reflect differences in product popularity, promotions, and demand conditions.

Therefore, we cannot infer missed demand directly from these aggregate averages. We will investigate the availability definition and evaluate any correction method before using it for forecasting.


In [151]:

import numpy as np
import pandas as pd

# Remove missing sales only for this analysis
analysis_df = df.dropna(subset=["sales"]).copy()

# Compare each product's sales with its own fully available days
full_availability_avg = (
    analysis_df.loc[analysis_df["availability"] == 1]
    .groupby("unique_id")["sales"]
    .mean()
    .rename("full_availability_avg_sales")
)

analysis_df = analysis_df.join(
    full_availability_avg,
    on="unique_id"
)

# Exclude products without a usable fully available baseline
analysis_df = analysis_df[
    analysis_df["full_availability_avg_sales"] > 0
].copy()

analysis_df["relative_sales"] = (
    analysis_df["sales"]
    / analysis_df["full_availability_avg_sales"]
)

# Limit extreme ratios for a descriptive comparison
analysis_df["relative_sales_capped"] = (
    analysis_df["relative_sales"].clip(upper=10)
)

availability_comparison = (
    analysis_df.groupby("availability_group", observed=True)
    .agg(
        observations=("sales", "size"),
        mean_relative_sales=("relative_sales_capped", "mean"),
        median_relative_sales=("relative_sales", "median")
    )
)

print("PRODUCT-NORMALIZED SALES BY AVAILABILITY")
print(availability_comparison.round(3).to_string())

print("\nPRODUCTS WITH FULL-AVAILABILITY BASELINE:")
print(analysis_df["unique_id"].nunique())


PRODUCT-NORMALIZED SALES BY AVAILABILITY
                    observations  mean_relative_sales  median_relative_sales
availability_group                                                          
1–25%                      61777                0.961                  0.828
25–50%                    162426                0.898                  0.812
50–75%                    269919                1.131                  0.960
75–99%                    476583                1.094                  0.972
100%                     3035159                0.998                  0.892

PRODUCTS WITH FULL-AVAILABILITY BASELINE:
5376



### Product-Level Availability Analysis

We compare recorded sales with each product's average sales on fully available days.

The analysis covers 5,376 product series with a usable fully available baseline.

Median relative sales are 0.828 for 1–25% availability, 0.812 for 25–50%, 0.960 for 50–75%, and 0.972 for 75–99%.

The relationship is not strictly monotonic, suggesting that availability alone does not explain all variation in sales.

This preliminary analysis is descriptive. Differences in demand, promotions, seasonality, and the timing of stockouts may influence these results.

Before estimating missed demand, we must verify the definition of the availability variable and evaluate the assumptions underlying any correction method.



### Official Dataset Documentation and Methodological Adjustment

The Rohlik competition documentation confirms that `availability` represents the proportion of the day a product was available, while `sales` has already been adjusted to represent full potential sales using Rohlik's internal methodology.

Therefore, dividing sales by availability would apply an inappropriate second correction.

We instead construct a semi-synthetic censored-sales series using:

synthetic_raw_sales = sales × availability

We compare models trained on synthetic censored sales against models trained on Rohlik-adjusted sales.

The adjusted sales represent an estimated demand reference, not independently verified true demand. Likewise, synthetic censored sales do not reproduce actual historical transactions.


## Step 13: Prepare Raw and Adjusted Sales

Rohlik's `sales` column already represents sales adjusted for product availability.

We construct an approximate censored-sales series by multiplying adjusted sales by the reported availability fraction.

- `adjusted_sales`: Rohlik's original, internally adjusted sales.
- `synthetic_raw_sales`: Simulated sales under partial availability.
- `availability`: Fraction of the day the product was available.

This is a semi-synthetic experiment. The synthetic values are not observed historical transactions.

We retain the original adjusted sales for evaluating the forecasting models. Missing sales observations are excluded before creating the modeling dataset.


In [152]:

import numpy as np
import pandas as pd

# Start from the Rohlik data we already loaded
rohlik = df[
    ["unique_id", "date", "warehouse",
     "sales", "availability", "sell_price_main"]
].copy()

# Remove rows where sales are missing
rows_before = len(rohlik)
rohlik = rohlik.dropna(subset=["sales"]).copy()

# Rename Rohlik's already-corrected sales
rohlik = rohlik.rename(
    columns={"sales": "adjusted_sales"}
)

# Construct simulated censored sales
rohlik["synthetic_raw_sales"] = (
    rohlik["adjusted_sales"] * rohlik["availability"]
)

# Sort by product and date
rohlik = rohlik.sort_values(
    ["unique_id", "date"]
).reset_index(drop=True)

print("Original rows:", rows_before)
print("Missing sales rows removed:", rows_before - len(rohlik))
print("Cleaned rows:", len(rohlik))

print("\nFIRST 5 ROWS:")
print(rohlik.head().to_string(index=False))

print("\nSALES COMPARISON:")
print(
    rohlik[
        ["adjusted_sales", "synthetic_raw_sales"]
    ].describe().round(2)
)

print("\nDATA VALIDATION:")
print(
    "Synthetic sales <= adjusted sales:",
    bool((
        rohlik["synthetic_raw_sales"]
        <= rohlik["adjusted_sales"]
    ).all())
)

print(
    "Negative synthetic sales:",
    int((rohlik["synthetic_raw_sales"] < 0).sum())
)

print(
    "Duplicate product-date rows:",
    int(rohlik.duplicated(["unique_id", "date"]).sum())
)


Original rows: 4007419
Missing sales rows removed: 52
Cleaned rows: 4007367

FIRST 5 ROWS:
 unique_id       date  warehouse  adjusted_sales  availability  sell_price_main  synthetic_raw_sales
         0 2022-07-18 Budapest_1            3.97          0.09           710.89               0.3573
         0 2022-07-19 Budapest_1           73.36          1.00           710.89              73.3600
         0 2022-07-20 Budapest_1          558.09          0.96           710.89             535.7664
         0 2022-07-21 Budapest_1           14.03          0.06           710.89               0.8418
         0 2022-07-22 Budapest_1          558.53          0.97           710.89             541.7741

SALES COMPARISON:
       adjusted_sales  synthetic_raw_sales
count      4007367.00           4007367.00
mean           108.38               101.08
std            360.33               343.63
min              0.00                 0.00
25%             17.98                15.99
50%             39.52     


## Step 13B: Chronological Train, Validation, and Test Splits

We divide the Rohlik dataset chronologically to avoid training on future observations.

To remain consistent with Stream 2, we use a 14-day validation period and a 7-day test period. All earlier observations are assigned to training.

Unlike FreshRetailNet, Rohlik provides several years of history, so we use dataset-specific calendar dates rather than copying Stream 2's original dates.

Both synthetic raw-sales and retailer-adjusted sales models will use identical splits.


In [153]:

import pandas as pd

# Use the final date with available training targets
last_date = rohlik["date"].max()

# Same validation/test durations as Stream 2
test_start = last_date - pd.Timedelta(days=6)
valid_start = test_start - pd.Timedelta(days=14)

# Assign chronological splits
rohlik["set"] = "train"

rohlik.loc[
    (rohlik["date"] >= valid_start) &
    (rohlik["date"] < test_start),
    "set"
] = "valid"

rohlik.loc[
    rohlik["date"] >= test_start,
    "set"
] = "test"

print("LAST AVAILABLE DATE:", last_date.date())
print("VALIDATION START:", valid_start.date())
print("TEST START:", test_start.date())

print("\nSPLIT SUMMARY:")
summary = (
    rohlik.groupby("set")
    .agg(
        rows=("unique_id", "size"),
        products=("unique_id", "nunique"),
        start_date=("date", "min"),
        end_date=("date", "max")
    )
)

print(summary.to_string())

print("\nDATA VALIDATION:")
print(
    "Missing split labels:",
    rohlik["set"].isna().sum()
)

print(
    "Duplicate product-date rows:",
    rohlik.duplicated(["unique_id", "date"]).sum()
)

print(
    "Total rows match:",
    summary["rows"].sum() == len(rohlik)
)


LAST AVAILABLE DATE: 2024-06-02
VALIDATION START: 2024-05-13
TEST START: 2024-05-27

SPLIT SUMMARY:
          rows  products start_date   end_date
set                                           
test     23762      3573 2024-05-27 2024-06-02
train  3936419      5382 2020-08-01 2024-05-12
valid    47186      3657 2024-05-13 2024-05-26

DATA VALIDATION:
Missing split labels: 0
Duplicate product-date rows: 0
Total rows match: True



## Step 14: Product History and Data Coverage

We examine product coverage across the chronological data splits and identify gaps in historical observations.

Because the Rohlik dataset contains missing dates for some product series, we verify that test products have sufficient historical information.

This check will inform our product selection and feature-engineering strategy before forecasting.

We retain the team's seven-day forecasting horizon and will ensure that future target values are not used as prediction features.


In [154]:

import pandas as pd

# Product identifiers in each split
train_ids = set(rohlik.loc[rohlik["set"] == "train", "unique_id"])
valid_ids = set(rohlik.loc[rohlik["set"] == "valid", "unique_id"])
test_ids = set(rohlik.loc[rohlik["set"] == "test", "unique_id"])

print("PRODUCT COVERAGE")
print("Training products:", len(train_ids))
print("Validation products:", len(valid_ids))
print("Test products:", len(test_ids))
print("Test products also in training:", len(test_ids & train_ids))
print("Test products missing from training:", len(test_ids - train_ids))
print("Products appearing in all three splits:",
      len(train_ids & valid_ids & test_ids))

# Historical observations per product
train_counts = (
    rohlik[rohlik["set"] == "train"]
    .groupby("unique_id")
    .size()
)

test_counts = (
    rohlik[rohlik["set"] == "test"]
    .groupby("unique_id")
    .size()
)

print("\nTRAINING HISTORY PER PRODUCT")
print(train_counts.describe().round(1).to_string())

print("\nTEST OBSERVATIONS PER PRODUCT")
print(test_counts.describe().round(1).to_string())

print("\nTEST PRODUCTS WITH SUFFICIENT HISTORY")
for minimum in [28, 60, 90]:
    eligible = train_counts[train_counts >= minimum].index
    print(
        f"At least {minimum} training rows:",
        len(test_ids & set(eligible))
    )

print("\nMISSING DATES WITHIN TEST PERIOD")
expected_test_dates = 7
incomplete_test = (test_counts < expected_test_dates).sum()

print("Products with fewer than 7 test observations:",
      incomplete_test)

print("Products with all 7 test observations:",
      (test_counts == expected_test_dates).sum())


PRODUCT COVERAGE
Training products: 5382
Validation products: 3657
Test products: 3573
Test products also in training: 3565
Test products missing from training: 8
Products appearing in all three splits: 3527

TRAINING HISTORY PER PRODUCT
count    5382.0
mean      731.4
std       490.0
min         1.0
25%       252.2
50%       719.0
75%      1265.8
max      1381.0

TEST OBSERVATIONS PER PRODUCT
count    3573.0
mean        6.7
std         1.0
min         1.0
25%         7.0
50%         7.0
75%         7.0
max         7.0

TEST PRODUCTS WITH SUFFICIENT HISTORY
At least 28 training rows: 3534
At least 60 training rows: 3509
At least 90 training rows: 3485

MISSING DATES WITHIN TEST PERIOD
Products with fewer than 7 test observations: 546
Products with all 7 test observations: 3027



## Step 15: Fixed Product Sample for Forecasting

We select a fixed set of Rohlik product series for the forecasting experiments.

To keep the comparison consistent with Stream 2, we target 2,000 series with sufficient training history.

Eligibility is based on historical coverage and the presence of complete validation and test observation dates.

Both synthetic censored-sales and retailer-adjusted sales models will use the same selected product series, chronological splits, and evaluation observations.

We will use a fixed random seed for reproducibility.


In [155]:

import numpy as np
import pandas as pd

SEED = 42
SAMPLE_SIZE = 2000

# Minimum historical training records
eligible_train = set(
    train_counts[train_counts >= 90].index
)

# Full 7-day test coverage
complete_test = set(
    test_counts[test_counts == 7].index
)

# Full 14-day validation coverage
valid_counts = (
    rohlik[rohlik["set"] == "valid"]
    .groupby("unique_id")
    .size()
)

complete_valid = set(
    valid_counts[valid_counts == 14].index
)

# Eligible product series
eligible_ids = sorted(
    eligible_train & complete_test & complete_valid
)

print("ELIGIBLE PRODUCT SERIES:", len(eligible_ids))

if len(eligible_ids) < SAMPLE_SIZE:
    print("WARNING: Fewer than 2,000 eligible series.")
else:
    rng = np.random.default_rng(SEED)

    selected_ids = rng.choice(
        eligible_ids,
        size=SAMPLE_SIZE,
        replace=False
    )

    selected_ids = sorted(selected_ids)

    rohlik_sample = rohlik[
        rohlik["unique_id"].isin(selected_ids)
    ].copy()

    print("\nSELECTED PRODUCTS:", len(selected_ids))

    print("\nSAMPLE SPLIT SUMMARY:")
    print(
        rohlik_sample.groupby("set")
        .agg(
            rows=("unique_id", "size"),
            products=("unique_id", "nunique")
        ).to_string()
    )

    print("\nDATA VALIDATION:")
    print("Duplicate product-date rows:",
          rohlik_sample.duplicated(["unique_id", "date"]).sum())

    print("Minimum validation rows per selected product:",
          valid_counts.loc[selected_ids].min())

    print("Minimum test rows per selected product:",
          test_counts.loc[selected_ids].min())


ELIGIBLE PRODUCT SERIES: 2334

SELECTED PRODUCTS: 2000

SAMPLE SPLIT SUMMARY:
          rows  products
set                     
test     14000      2000
train  2085365      2000
valid    28000      2000

DATA VALIDATION:
Duplicate product-date rows: 0
Minimum validation rows per selected product: 14
Minimum test rows per selected product: 7



## Step 16: Sample Representativeness

We compare our selected 2,000 product series against the eligible Rohlik product population.

We evaluate warehouse representation and historical partial-availability rates to determine whether the sample adequately covers different operating conditions.

We calculate stockout-related characteristics using training observations only.

This check helps ensure that our cross-retailer experiment is not dominated by a particular warehouse or availability pattern.


In [156]:

import pandas as pd
import numpy as np

# Use training observations only to measure availability patterns
train_data = rohlik[
    (rohlik["set"] == "train") &
    (rohlik["unique_id"].isin(eligible_ids))
]

product_profile = (
    train_data.groupby("unique_id")
    .agg(
        warehouse=("warehouse", "first"),
        partial_availability_rate=(
            "availability",
            lambda x: (x < 1).mean()
        )
    )
)

product_profile["sample"] = np.where(
    product_profile.index.isin(selected_ids),
    "Selected",
    "Not selected"
)

print("WAREHOUSE REPRESENTATION (%)")

warehouse_comparison = pd.DataFrame({
    "Eligible": product_profile["warehouse"]
        .value_counts(normalize=True) * 100,
    "Selected": product_profile.loc[
        product_profile["sample"] == "Selected",
        "warehouse"
    ].value_counts(normalize=True) * 100
}).fillna(0)

print(warehouse_comparison.round(2).to_string())

# Stockout groups based only on historical training data
product_profile["stockout_group"] = pd.cut(
    product_profile["partial_availability_rate"],
    bins=[-0.001, 0.10, 0.30, 1.0],
    labels=["Low (0-10%)", "Medium (10-30%)", "High (>30%)"]
)

print("\nSTOCKOUT GROUP REPRESENTATION (%)")

stockout_comparison = pd.DataFrame({
    "Eligible": product_profile["stockout_group"]
        .value_counts(normalize=True) * 100,
    "Selected": product_profile.loc[
        product_profile["sample"] == "Selected",
        "stockout_group"
    ].value_counts(normalize=True) * 100
}).fillna(0)

print(stockout_comparison.round(2).to_string())

print("\nAVERAGE PARTIAL-AVAILABILITY RATE:")
print(
    "Eligible:",
    round(product_profile["partial_availability_rate"].mean() * 100, 2),
    "%"
)
print(
    "Selected:",
    round(
        product_profile.loc[
            product_profile["sample"] == "Selected",
            "partial_availability_rate"
        ].mean() * 100, 2
    ),
    "%"
)


WAREHOUSE REPRESENTATION (%)
             Eligible  Selected
warehouse                      
Prague_1        20.48     20.40
Prague_3        20.18     20.25
Prague_2        19.24     19.30
Brno_1          15.38     15.50
Budapest_1      11.01     10.90
Munich_1         7.63      7.75
Frankfurt_1      6.08      5.90

STOCKOUT GROUP REPRESENTATION (%)
                 Eligible  Selected
stockout_group                     
Medium (10-30%)     42.76     42.95
Low (0-10%)         32.52     32.00
High (>30%)         24.72     25.05

AVERAGE PARTIAL-AVAILABILITY RATE:
Eligible: 24.41 %
Selected: 24.56 %



### Sample Selection Findings

We identified 2,334 eligible product series and selected 2,000 using random seed 42.

The sample closely matches the eligible population across all seven warehouses and historical partial-availability groups.

The selected sample has a mean partial-availability rate of 24.56%, compared with 24.41% in the eligible population.

All selected products have at least 90 historical training observations and complete 14-day validation and 7-day test coverage.

We retain this fixed sample for both forecasting conditions to ensure consistent comparisons.

Because eligibility requires complete validation and test coverage, our sample may not fully represent products with irregular observations.



## Step 17: Save the Fixed Stream 3 Dataset

We save our selected Rohlik product identifiers and modeling dataset to the shared Google Drive.

These files preserve the fixed sample and chronological splits for subsequent forecasting experiments.

The modeling dataset includes retailer-adjusted sales and semi-synthetic censored sales. We will add calendar information and forecasting features in later steps.


In [157]:

from pathlib import Path
import pandas as pd

# Existing shared processed-data folder
STREAM3_DIR = PROCESSED_DIR

# New Stream 3 files only — do not overwrite teammates' files
IDS_PATH = STREAM3_DIR / "stream3_rohlik_sample_ids_v1.csv"
DATA_PATH = STREAM3_DIR / "stream3_rohlik_daily_v1.parquet"

# Save selected product identifiers
sample_ids_df = pd.DataFrame({
    "unique_id": selected_ids
})

sample_ids_df.to_csv(IDS_PATH, index=False)

# Save selected Rohlik observations and existing splits
columns_to_save = [
    "unique_id",
    "date",
    "warehouse",
    "adjusted_sales",
    "synthetic_raw_sales",
    "availability",
    "sell_price_main",
    "set"
]

rohlik_sample[columns_to_save].to_parquet(
    DATA_PATH,
    index=False
)

print("FILES SAVED SUCCESSFULLY")

for path in [IDS_PATH, DATA_PATH]:
    print(
        path.name,
        "|",
        round(path.stat().st_size / 1024**2, 2),
        "MB"
    )

print("\nVERIFY SAVED FILES")

saved_ids = pd.read_csv(IDS_PATH)
saved_data = pd.read_parquet(DATA_PATH)

print("Saved product IDs:", len(saved_ids))
print("Saved dataset rows:", len(saved_data))
print("Saved dataset columns:", saved_data.columns.tolist())

print("\nSPLIT COUNTS:")
print(saved_data["set"].value_counts())


FILES SAVED SUCCESSFULLY
stream3_rohlik_sample_ids_v1.csv | 0.01 MB
stream3_rohlik_daily_v1.parquet | 18.25 MB

VERIFY SAVED FILES
Saved product IDs: 2000
Saved dataset rows: 2127365
Saved dataset columns: ['unique_id', 'date', 'warehouse', 'adjusted_sales', 'synthetic_raw_sales', 'availability', 'sell_price_main', 'set']

SPLIT COUNTS:
set
train    2085365
valid      28000
test       14000
Name: count, dtype: int64



## Step 18: Enrich the Rohlik Dataset with Product and Calendar Features

We join Rohlik's product metadata and calendar information to the fixed 2,000-series sample.

Product metadata provides category and warehouse context, while calendar features capture holidays, school holidays, and warehouse closures.

These features will help us adapt the forecasting methodology from Stream 2 while preserving the same product sample and chronological splits.

We verify join uniqueness and row counts to avoid unintentionally duplicating observations.


In [158]:

import pandas as pd

# Load product metadata and calendar
inventory = pd.read_csv(ROHLIK_DIR / "inventory.csv")

calendar = pd.read_csv(
    ROHLIK_DIR / "calendar.csv",
    parse_dates=["date"]
)

print("INVENTORY SHAPE:", inventory.shape)
print("CALENDAR SHAPE:", calendar.shape)

print("\nJOIN KEY VALIDATION:")
print(
    "Duplicate inventory unique_id:",
    inventory.duplicated(["unique_id"]).sum()
)
print(
    "Duplicate calendar warehouse-date:",
    calendar.duplicated(["warehouse", "date"]).sum()
)

# Work from the saved Stream 3 dataset
rohlik_enriched = pd.read_parquet(DATA_PATH)
original_rows = len(rohlik_enriched)

# Add product metadata
product_cols = [
    "unique_id",
    "product_unique_id",
    "L1_category_name_en",
    "L2_category_name_en"
]

rohlik_enriched = rohlik_enriched.merge(
    inventory[product_cols],
    on="unique_id",
    how="left",
    validate="many_to_one"
)

# Add warehouse-specific calendar
calendar_cols = [
    "warehouse",
    "date",
    "holiday",
    "shops_closed",
    "winter_school_holidays",
    "school_holidays"
]

rohlik_enriched = rohlik_enriched.merge(
    calendar[calendar_cols],
    on=["warehouse", "date"],
    how="left",
    validate="many_to_one"
)

print("\nENRICHED DATASET:")
print("Original rows:", original_rows)
print("Enriched rows:", len(rohlik_enriched))
print("Rows preserved:", len(rohlik_enriched) == original_rows)

print("\nMISSING JOINED FEATURES (%):")
new_cols = product_cols[1:] + calendar_cols[2:]
print(
    (rohlik_enriched[new_cols].isna().mean() * 100)
    .round(2)
    .to_string()
)

print("\nFIRST 3 ROWS:")
print(rohlik_enriched.head(3).to_string(index=False))


INVENTORY SHAPE: (5432, 8)
CALENDAR SHAPE: (23016, 7)

JOIN KEY VALIDATION:
Duplicate inventory unique_id: 0
Duplicate calendar warehouse-date: 0

ENRICHED DATASET:
Original rows: 2127365
Enriched rows: 2127365
Rows preserved: True

MISSING JOINED FEATURES (%):
product_unique_id         0.0
L1_category_name_en       0.0
L2_category_name_en       0.0
holiday                   0.0
shops_closed              0.0
winter_school_holidays    0.0
school_holidays           0.0

FIRST 3 ROWS:
 unique_id       date warehouse  adjusted_sales  synthetic_raw_sales  availability  sell_price_main   set  product_unique_id L1_category_name_en L2_category_name_en  holiday  shops_closed  winter_school_holidays  school_holidays
         9 2020-08-01  Prague_1           21.15               8.8830          0.42            27.45 train                  7              Bakery         Bakery_L2_5        0             0                       0                0
         9 2020-08-02  Prague_1            7.94        


## Step 19: Save Enriched Modeling Dataset

We verified that the product and calendar joins preserve all 2,127,365 observations without missing metadata.

We save the enriched dataset separately from the original fixed sample to preserve reproducibility.

The enriched dataset will serve as the input for our Seasonal Naive and LightGBM forecasting experiments.


In [159]:

# Save enriched Stream 3 dataset
ENRICHED_PATH = PROCESSED_DIR / "stream3_rohlik_enriched_v1.parquet"

rohlik_enriched.to_parquet(
    ENRICHED_PATH,
    index=False
)

# Verify saved file
saved_enriched = pd.read_parquet(ENRICHED_PATH)

print("FILE SAVED:", ENRICHED_PATH.name)
print("FILE SIZE (MB):", round(ENRICHED_PATH.stat().st_size / 1024**2, 2))

print("\nDATA VALIDATION:")
print("Rows:", len(saved_enriched))
print("Products:", saved_enriched["unique_id"].nunique())
print("Columns:", len(saved_enriched.columns))
print("Duplicate product-date rows:",
      saved_enriched.duplicated(["unique_id", "date"]).sum())

print("\nSPLIT COUNTS:")
print(saved_enriched["set"].value_counts())


FILE SAVED: stream3_rohlik_enriched_v1.parquet
FILE SIZE (MB): 18.49

DATA VALIDATION:
Rows: 2127365
Products: 2000
Columns: 15
Duplicate product-date rows: 0

SPLIT COUNTS:
set
train    2085365
valid      28000
test       14000
Name: count, dtype: int64



## Step 20: Review Stream 2 Forecasting Methodology

We inspect the existing Stream 2 forecasting notebook before adapting its models to Rohlik.

Our objective is to reproduce its main modeling choices, including Seasonal Naive and LightGBM, lag features, validation strategy, and seven-day forecast horizon.

We will adapt dataset-specific features while ensuring that models do not use future observed sales or future availability values.


In [160]:

from pathlib import Path
import nbformat

# Locate teammate's original forecasting notebook
STREAM2_PATH = NOTEBOOK_DIR / "stream2_forecasting.ipynb"

print("Notebook found:", STREAM2_PATH.exists())

if STREAM2_PATH.exists():
    nb = nbformat.read(str(STREAM2_PATH), as_version=4)

    keywords = [
        "def build_features",
        "def score_table",
        "def keep_columns",
        "LGBMRegressor",
        "seasonal_naive",
        "early_stopping",
        "shift(",
        "rolling("
    ]

    print("Total notebook cells:", len(nb.cells))

    for i, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue

        matched = [
            word for word in keywords
            if word.lower() in cell.source.lower()
        ]

        if matched:
            print("\n" + "=" * 55)
            print(f"CELL {i} | Keywords: {', '.join(matched)}")
            print("=" * 55)
            print(cell.source[:6500])


Notebook found: True
Total notebook cells: 40

CELL 9 | Keywords: def build_features, shift(, rolling(
def build_features(df, value_col):
    """
    Make one version of the modeling table.
    value_col is 'sales' (the raw version) or 'demand' (the corrected version).
    """
    d = df.copy()
    d['target'] = d[value_col]          # the thing the model learns to predict
    g = d.groupby('series_id')['target']

    # Sales exactly 1, 2 and 3 weeks ago (same weekday, so weekly patterns carry over)
    d['lag7']  = g.shift(7)
    d['lag14'] = g.shift(14)
    d['lag21'] = g.shift(21)

    # Average of the most recent 7 / 14 days we're allowed to see (starting 7 days back)
    d['roll7']  = g.transform(lambda s: s.shift(7).rolling(7,  min_periods=3).mean())
    d['roll14'] = g.transform(lambda s: s.shift(7).rolling(14, min_periods=7).mean())

    # Average of the same weekday over the last 3 weeks (a steadier version of lag7)
    d['dow_avg3'] = d[['lag7', 'lag14', 'lag21']].mean(axis=1


## Step 21: Historical Date Continuity

We examine missing calendar dates within each selected Rohlik product series before constructing lag and rolling features.

Stream 2 uses seven-, fourteen-, and twenty-one-day lag features. For Rohlik, we must ensure these refer to actual calendar days rather than simply previous recorded rows.

We will also prevent validation and test target values from being used as future forecasting inputs.


In [161]:

import pandas as pd

# Load our saved enriched dataset
model_base = pd.read_parquet(ENRICHED_PATH)

# Sort product observations chronologically
model_base = model_base.sort_values(
    ["unique_id", "date"]
).reset_index(drop=True)

# Identify gaps between successive dates per product
model_base["previous_date"] = (
    model_base.groupby("unique_id")["date"].shift(1)
)

model_base["days_since_previous"] = (
    model_base["date"] - model_base["previous_date"]
).dt.days

gaps = model_base[
    model_base["days_since_previous"] > 1
].copy()

print("HISTORICAL DATE CONTINUITY")
print("Products in sample:", model_base["unique_id"].nunique())
print("Products with at least one date gap:",
      gaps["unique_id"].nunique())
print("Total gaps:", len(gaps))

print("\nGAP LENGTH DISTRIBUTION:")
print(
    gaps["days_since_previous"]
    .describe()
    .round(2)
    .to_string()
)

print("\nGAPS BY SPLIT:")
print(gaps["set"].value_counts().to_string())

print("\nEXAMPLE GAPS:")
print(
    gaps[
        ["unique_id", "date",
         "previous_date", "days_since_previous"]
    ].head(10).to_string(index=False)
)


HISTORICAL DATE CONTINUITY
Products in sample: 2000
Products with at least one date gap: 1971
Total gaps: 28252

GAP LENGTH DISTRIBUTION:
count    28252.00
mean         7.44
std         32.23
min          2.00
25%          2.00
50%          2.00
75%          3.00
max        899.00

GAPS BY SPLIT:
set
train    28228
valid       24

EXAMPLE GAPS:
 unique_id       date previous_date  days_since_previous
         9 2020-08-31    2020-08-29                  2.0
         9 2020-09-29    2020-09-27                  2.0
         9 2020-10-24    2020-10-21                  3.0
         9 2020-10-30    2020-10-28                  2.0
         9 2020-11-22    2020-11-20                  2.0
         9 2020-12-12    2020-12-10                  2.0
         9 2021-04-06    2021-04-04                  2.0
         9 2021-04-19    2021-04-17                  2.0
         9 2021-05-03    2021-05-01                  2.0
         9 2021-05-10    2021-05-08                  2.0



## Step 22: Calendar-Aligned Forecasting Features

We adapt Stream 2's lag and rolling features to Rohlik's irregular daily observations.

Instead of shifting by observation count, we construct historical features using calendar dates.

Our forecasting design uses:
- Lagged sales from 7, 14, and 21 calendar days earlier.
- Historical rolling averages.
- Day-of-week and known calendar information.
- Training-period product averages.

We preserve missing historical observations as missing values rather than treating them as zero sales.

To prevent forecasting leakage, validation forecasts use information available before validation begins, and test forecasts use information available before the seven-day test period begins.


In [162]:

import numpy as np
import pandas as pd

# Start with the fixed enriched dataset
forecast_base = pd.read_parquet(ENRICHED_PATH)
forecast_base = forecast_base.sort_values(
    ["unique_id", "date"]
).reset_index(drop=True)

# Calendar features known in advance
forecast_base["day_of_week"] = (
    forecast_base["date"].dt.dayofweek
)

forecast_base["month"] = (
    forecast_base["date"].dt.month
)

# Create features separately for each forecasting version
def build_rohlik_features(df, target_col):
    d = df.copy()
    d["target"] = d[target_col]

    # Product-level mean using training data only
    train_means = (
        d.loc[d["set"] == "train"]
        .groupby("unique_id")["target"]
        .mean()
    )
    d["train_mean"] = d["unique_id"].map(train_means)

    # Build a date-based lookup of historical targets
    history = d[
        ["unique_id", "date", "target"]
    ].copy()

    # Each lag is an exact calendar-day offset
    for lag in [7, 14, 21]:
        lagged = history.copy()
        lagged["date"] = (
            lagged["date"] + pd.Timedelta(days=lag)
        )
        lagged = lagged.rename(
            columns={"target": f"lag{lag}"}
        )

        d = d.merge(
            lagged,
            on=["unique_id", "date"],
            how="left",
            validate="one_to_one"
        )

    d["dow_avg3"] = d[
        ["lag7", "lag14", "lag21"]
    ].mean(axis=1)

    return d

raw_features = build_rohlik_features(
    forecast_base,
    "synthetic_raw_sales"
)

corrected_features = build_rohlik_features(
    forecast_base,
    "adjusted_sales"
)

print("FEATURE TABLES")
print("Raw rows:", len(raw_features))
print("Corrected rows:", len(corrected_features))

print("\nMISSING LAG FEATURES (%):")
for name, frame in [
    ("Raw", raw_features),
    ("Corrected", corrected_features)
]:
    print(
        name,
        (frame[["lag7", "lag14", "lag21"]].isna().mean() * 100)
        .round(2)
        .to_dict()
    )

print("\nEXAMPLE FEATURES:")
print(
    raw_features[
        ["unique_id", "date", "set",
         "target", "lag7", "lag14",
         "lag21", "dow_avg3", "train_mean"]
    ].head(10).to_string(index=False)
)


FEATURE TABLES
Raw rows: 2127365
Corrected rows: 2127365

MISSING LAG FEATURES (%):
Raw {'lag7': 3.02, 'lag14': 4.13, 'lag21': 5.09}
Corrected {'lag7': 3.02, 'lag14': 4.13, 'lag21': 5.09}

EXAMPLE FEATURES:
 unique_id       date   set  target    lag7  lag14  lag21  dow_avg3  train_mean
         9 2020-08-01 train  8.8830     NaN    NaN    NaN       NaN   44.955624
         9 2020-08-02 train  4.8434     NaN    NaN    NaN       NaN   44.955624
         9 2020-08-03 train 10.7967     NaN    NaN    NaN       NaN   44.955624
         9 2020-08-04 train 21.7100     NaN    NaN    NaN       NaN   44.955624
         9 2020-08-05 train 28.4900     NaN    NaN    NaN       NaN   44.955624
         9 2020-08-06 train 34.3629     NaN    NaN    NaN       NaN   44.955624
         9 2020-08-07 train 11.8710     NaN    NaN    NaN       NaN   44.955624
         9 2020-08-08 train 20.8300  8.8830    NaN    NaN    8.8830   44.955624
         9 2020-08-09 train  5.7500  4.8434    NaN    NaN    4.8434   44.


## Step 23: Prevent Forecasting Data Leakage

We ensure that our forecasting features only use historical sales available at the forecast origin.

Forecast origins:
- Validation: May 12, 2024
- Test: May 26, 2024

Our validation period spans 14 days. For its second week, we must use older historical observations instead of actual sales from the first validation week.

We construct leakage-safe lag features for both synthetic raw sales and retailer-adjusted sales.

Missing historical observations remain missing rather than being replaced with zero.


In [163]:

# Forecast origins: last date known before each evaluation period
VALID_ORIGIN = pd.Timestamp("2024-05-12")
TEST_ORIGIN = pd.Timestamp("2024-05-26")

def make_safe_lags(frame, value_col):
    d = frame.copy()
    d = d.sort_values(["unique_id", "date"]).reset_index(drop=True)

    history = d[["unique_id", "date", value_col]].copy()

    # Exact calendar-day lookups
    for lag in [7, 14, 21, 28]:
        lookup = history.copy()
        lookup["date"] += pd.Timedelta(days=lag)
        lookup = lookup.rename(
            columns={value_col: f"source_lag{lag}"}
        )

        d = d.merge(
            lookup,
            on=["unique_id", "date"],
            how="left",
            validate="one_to_one"
        )

    # First validation week can use lag7.
    # Second validation week must start at lag14.
    second_valid_week = (
        (d["set"] == "valid") &
        (d["date"] >= pd.Timestamp("2024-05-20"))
    )

    d["lag7_safe"] = d["source_lag7"]
    d["lag14_safe"] = d["source_lag14"]
    d["lag21_safe"] = d["source_lag21"]

    d.loc[second_valid_week, "lag7_safe"] = (
        d.loc[second_valid_week, "source_lag14"]
    )
    d.loc[second_valid_week, "lag14_safe"] = (
        d.loc[second_valid_week, "source_lag21"]
    )
    d.loc[second_valid_week, "lag21_safe"] = (
        d.loc[second_valid_week, "source_lag28"]
    )

    d["dow_avg3_safe"] = d[
        ["lag7_safe", "lag14_safe", "lag21_safe"]
    ].mean(axis=1)

    return d

raw_safe = make_safe_lags(
    forecast_base,
    "synthetic_raw_sales"
)

corrected_safe = make_safe_lags(
    forecast_base,
    "adjusted_sales"
)

print("LEAKAGE-SAFE FEATURE TABLES")
print("Raw rows:", len(raw_safe))
print("Corrected rows:", len(corrected_safe))

print("\nVALIDATION: FIRST AND SECOND WEEK EXAMPLES")
print(
    raw_safe.loc[
        (raw_safe["unique_id"] == selected_ids[0]) &
        (raw_safe["set"] == "valid"),
        ["date", "source_lag7", "source_lag14",
         "lag7_safe", "lag14_safe"]
    ].to_string(index=False)
)

print("\nMISSING SAFE LAG7:")
for split in ["train", "valid", "test"]:
    section = raw_safe[raw_safe["set"] == split]
    print(
        split,
        round(section["lag7_safe"].isna().mean() * 100, 2),
        "%"
    )

print("\nROW VALIDATION:")
print(
    "Rows preserved:",
    len(raw_safe) == len(forecast_base)
    and len(corrected_safe) == len(forecast_base)
)


LEAKAGE-SAFE FEATURE TABLES
Raw rows: 2127365
Corrected rows: 2127365

VALIDATION: FIRST AND SECOND WEEK EXAMPLES
      date  source_lag7  source_lag14  lag7_safe  lag14_safe
2024-05-13      20.0300         37.45      20.03       37.45
2024-05-14      63.3500         43.64      63.35       43.64
2024-05-15      35.3700         19.70      35.37       19.70
2024-05-16      53.7700         31.10      53.77       31.10
2024-05-17      51.9700         38.39      51.97       38.39
2024-05-18      21.6000         39.28      21.60       39.28
2024-05-19      66.9500         34.15      66.95       34.15
2024-05-20      31.5400         20.03      20.03       37.45
2024-05-21      39.2600         63.35      63.35       43.64
2024-05-22      52.0008         35.37      35.37       19.70
2024-05-23      43.1900         53.77      53.77       31.10
2024-05-24      57.9900         51.97      51.97       38.39
2024-05-25      30.1100         21.60      21.60       39.28
2024-05-26      28.2900         


## Step 24: Final Forecasting Feature Construction

We adapt Stream 2's LightGBM feature engineering to Rohlik.

We use seven-day forecasting blocks, calendar-aligned historical lags, rolling averages, training-period product means, and known calendar variables.

Each validation and test prediction uses only information available before the beginning of its seven-day forecasting block.

We build separate feature tables for synthetic censored sales and retailer-adjusted sales, using identical observations and feature definitions.


In [164]:

import numpy as np
import pandas as pd

# Forecast each seven-day block from the preceding Sunday
def build_final_features(base, value_col):
    d = base.copy().sort_values(
        ["unique_id", "date"]
    ).reset_index(drop=True)

    d["target"] = d[value_col]
    d["day_of_week"] = d["date"].dt.dayofweek
    d["month"] = d["date"].dt.month

    # The start of each Monday-Sunday forecasting block
    d["week_start"] = (
        d["date"] -
        pd.to_timedelta(d["day_of_week"], unit="D")
    )

    d["forecast_origin"] = (
        d["week_start"] - pd.Timedelta(days=1)
    )

    # Calendar-aligned history (missing dates remain missing)
    history = d[["unique_id", "date", "target"]].copy()

    # Lags are measured from the forecast origin,
    # not from the prediction date.
    d["horizon"] = (
        d["date"] - d["forecast_origin"]
    ).dt.days

    # Same weekday in previous weeks
    for lag in [7, 14, 21]:
        lookup = history.rename(
            columns={
                "date": "history_date",
                "target": f"lag{lag}"
            }
        )

        d[f"history_date_{lag}"] = (
            d["date"] - pd.Timedelta(days=lag)
        )

        d = d.merge(
            lookup,
            left_on=["unique_id", f"history_date_{lag}"],
            right_on=["unique_id", "history_date"],
            how="left",
            validate="many_to_one"
        )

        d.loc[
            d[f"history_date_{lag}"] > d["forecast_origin"],
            f"lag{lag}"
        ] = np.nan

        d = d.drop(
            columns=["history_date", f"history_date_{lag}"]
        )

    # Historical weekly averages calculated at the origin
    # We use exact calendar-date lookups, not observation shifts.
    for window in [7, 14]:
        hist = history.copy()

        # Average historical observations in a rolling window,
        # evaluated at each forecast origin
        hist = hist.sort_values(["unique_id", "date"])

        # Build a complete daily grid for correct date alignment
        # Only for the last 28 days before each origin:
        # handled using date-filtered historical windows below.
        pass

    # Training-only product averages
    train_mean = (
        d.loc[d["set"] == "train"]
        .groupby("unique_id")["target"].mean()
    )
    d["train_mean"] = d["unique_id"].map(train_mean)

    d["dow_avg3"] = d[
        ["lag7", "lag14", "lag21"]
    ].mean(axis=1)

    d["holiday_flag"] = d["holiday"]
    d["shops_closed_flag"] = d["shops_closed"]

    return d

# We will finish the historical rolling averages
# before training the models.
raw_final = build_final_features(
    forecast_base, "synthetic_raw_sales"
)
corrected_final = build_final_features(
    forecast_base, "adjusted_sales"
)

print("FINAL FEATURE TABLES")
print("Raw rows:", len(raw_final))
print("Corrected rows:", len(corrected_final))

print("\nEXAMPLE TEST FEATURES:")
print(
    raw_final.loc[
        raw_final["set"] == "test",
        ["unique_id", "date", "forecast_origin",
         "horizon", "lag7", "lag14",
         "lag21", "train_mean"]
    ].head(7).to_string(index=False)
)

print("\nVALIDATION AND TEST LAG7 COVERAGE:")
for split in ["valid", "test"]:
    s = raw_final[raw_final["set"] == split]
    print(split, round(s["lag7"].notna().mean() * 100, 2), "%")


FINAL FEATURE TABLES
Raw rows: 2127365
Corrected rows: 2127365

EXAMPLE TEST FEATURES:
 unique_id       date forecast_origin  horizon  lag7   lag14  lag21  train_mean
         9 2024-05-27      2024-05-26        1 40.32 31.5400  20.03   44.955624
         9 2024-05-28      2024-05-26        2 42.42 39.2600  63.35   44.955624
         9 2024-05-29      2024-05-26        3 38.88 52.0008  35.37   44.955624
         9 2024-05-30      2024-05-26        4 47.87 43.1900  53.77   44.955624
         9 2024-05-31      2024-05-26        5 49.99 57.9900  51.97   44.955624
         9 2024-06-01      2024-05-26        6 35.62 30.1100  21.60   44.955624
         9 2024-06-02      2024-05-26        7 17.41 28.2900  66.95   44.955624

VALIDATION AND TEST LAG7 COVERAGE:
valid 99.07 %
test 100.0 %



## Step 25: Finalize Leakage-Safe Model Features

We complete Stream 2's forecasting features by adding historical 7-day and 14-day rolling averages.

Rolling features are calculated using observations available on or before the forecast origin. Missing historical dates are not assumed to represent zero demand.

We also replace the full-training-period product mean on training rows with a past-only expanding mean. Validation and test rows use the fixed product mean calculated from training observations.

This prevents future target values from leaking into model features.


In [165]:

import numpy as np
import pandas as pd

def complete_model_features(frame):
    d = frame.copy()

    # 1. Past-only product means for training rows
    train_mask = d["set"] == "train"

    train = d.loc[train_mask].sort_values(
        ["unique_id", "date"]
    ).copy()

    g = train.groupby("unique_id")["target"]

    train["past_mean"] = (
        (g.cumsum() - train["target"])
        / g.cumcount().replace(0, np.nan)
    )

    d["train_mean"] = np.nan
    d.loc[train.index, "train_mean"] = train["past_mean"]

    # Validation/test can use all training observations
    full_train_mean = (
        train.groupby("unique_id")["target"].mean()
    )

    holdout_mask = ~train_mask
    d.loc[holdout_mask, "train_mean"] = (
        d.loc[holdout_mask, "unique_id"]
        .map(full_train_mean)
    )

    # 2. Calendar-day rolling means at the forecast origin
    # Use a daily calendar grid so missing dates stay missing.
    history = d[
        ["unique_id", "date", "target"]
    ].copy()

    parts = []

    for uid, group in history.groupby("unique_id", sort=False):
        group = group.sort_values("date").set_index("date")

        daily = group[["target"]].asfreq("D")

        daily["roll7"] = (
            daily["target"].rolling(7, min_periods=3).mean()
        )

        daily["roll14"] = (
            daily["target"].rolling(14, min_periods=7).mean()
        )

        daily["unique_id"] = uid

        parts.append(
            daily[["unique_id", "roll7", "roll14"]]
            .reset_index()
            .rename(columns={"date": "forecast_origin"})
        )

    rolling_lookup = pd.concat(parts, ignore_index=True)

    d = d.merge(
        rolling_lookup,
        on=["unique_id", "forecast_origin"],
        how="left",
        validate="many_to_one"
    )

    # 3. Final model feature list
    feature_cols = [
        "lag7", "lag14", "lag21",
        "roll7", "roll14", "dow_avg3",
        "train_mean", "day_of_week", "month",
        "holiday", "shops_closed",
        "winter_school_holidays", "school_holidays"
    ]

    return d, feature_cols


raw_model, FEATURES_ROHLIK = complete_model_features(raw_final)
corrected_model, _ = complete_model_features(corrected_final)

print("FINAL MODEL TABLES")
print("Raw rows:", len(raw_model))
print("Corrected rows:", len(corrected_model))

print("\nFEATURES:")
print(FEATURES_ROHLIK)

print("\nTEST FEATURE MISSINGNESS (%):")
test_rows = raw_model[raw_model["set"] == "test"]
print(
    (test_rows[FEATURES_ROHLIK].isna().mean() * 100)
    .round(2).to_string()
)

print("\nEXAMPLE TEST ROWS:")
print(
    test_rows[
        ["unique_id", "date", "forecast_origin",
         "lag7", "roll7", "roll14", "train_mean"]
    ].head(7).round(2).to_string(index=False)
)

print("\nVALIDATION:")
print("Raw rows preserved:", len(raw_model) == len(raw_final))
print(
    "Corrected rows preserved:",
    len(corrected_model) == len(corrected_final)
)


FINAL MODEL TABLES
Raw rows: 2127365
Corrected rows: 2127365

FEATURES:
['lag7', 'lag14', 'lag21', 'roll7', 'roll14', 'dow_avg3', 'train_mean', 'day_of_week', 'month', 'holiday', 'shops_closed', 'winter_school_holidays', 'school_holidays']

TEST FEATURE MISSINGNESS (%):
lag7                      0.00
lag14                     0.00
lag21                     1.86
roll7                     0.00
roll14                    0.00
dow_avg3                  0.00
train_mean                0.00
day_of_week               0.00
month                     0.00
holiday                   0.00
shops_closed              0.00
winter_school_holidays    0.00
school_holidays           0.00

EXAMPLE TEST ROWS:
 unique_id       date forecast_origin  lag7  roll7  roll14  train_mean
         9 2024-05-27      2024-05-26 40.32  38.93   39.64       44.96
         9 2024-05-28      2024-05-26 42.42  38.93   39.64       44.96
         9 2024-05-29      2024-05-26 38.88  38.93   39.64       44.96
         9 2024-05-30 


## Step 26: Align Training Means with Forecast Origins

We ensure that the product-level average used for each training prediction excludes all observations from the prediction's own seven-day forecasting block.

For each training week, we calculate the product's mean using only previous training weeks.

Validation and test rows continue using the fixed training-period product mean.

This ensures that training features respect the same seven-day forecast-origin rule as our evaluation features.


In [166]:

import numpy as np
import pandas as pd

def fix_weekly_train_mean(frame):
    d = frame.copy()

    train = d.loc[d["set"] == "train"]

    # Summarize observations by product and forecasting week
    weekly = (
        train.groupby(
            ["unique_id", "week_start"],
            as_index=False
        )
        .agg(
            week_sum=("target", "sum"),
            week_count=("target", "count")
        )
        .sort_values(["unique_id", "week_start"])
    )

    g = weekly.groupby("unique_id")

    # Cumulative history EXCLUDING the current week
    weekly["prior_sum"] = (
        g["week_sum"].cumsum() - weekly["week_sum"]
    )
    weekly["prior_count"] = (
        g["week_count"].cumsum() - weekly["week_count"]
    )

    weekly["safe_train_mean"] = (
        weekly["prior_sum"] /
        weekly["prior_count"].replace(0, np.nan)
    )

    lookup = weekly[
        ["unique_id", "week_start", "safe_train_mean"]
    ]

    d = d.merge(
        lookup,
        on=["unique_id", "week_start"],
        how="left",
        validate="many_to_one"
    )

    # Replace training means only
    train_mask = d["set"] == "train"

    d.loc[train_mask, "train_mean"] = (
        d.loc[train_mask, "safe_train_mean"]
    )

    d = d.drop(columns=["safe_train_mean"])

    return d


raw_model = fix_weekly_train_mean(raw_model)
corrected_model = fix_weekly_train_mean(corrected_model)

print("ROWS:")
print("Raw:", len(raw_model))
print("Corrected:", len(corrected_model))

print("\nTRAINING MEAN MISSINGNESS (%):")
for name, data in [
    ("Raw", raw_model),
    ("Corrected", corrected_model)
]:
    train = data[data["set"] == "train"]

    print(
        name,
        round(train["train_mean"].isna().mean() * 100, 2),
        "%"
    )

print("\nTRAINING WEEK CONSISTENCY:")
example = raw_model[
    (raw_model["unique_id"] == selected_ids[0]) &
    (raw_model["set"] == "train")
].copy()

example = example[example["train_mean"].notna()]

print(
    example[
        ["date", "week_start", "forecast_origin",
         "target", "train_mean"]
    ].head(14).round(2).to_string(index=False)
)

print("\nVALIDATION:")
print(
    "All rows preserved:",
    len(raw_model) == len(forecast_base)
    and len(corrected_model) == len(forecast_base)
)
print(
    "Duplicate raw product-date rows:",
    raw_model.duplicated(["unique_id", "date"]).sum()
)


ROWS:
Raw: 2127365
Corrected: 2127365

TRAINING MEAN MISSINGNESS (%):
Raw 0.31 %
Corrected 0.31 %

TRAINING WEEK CONSISTENCY:
      date week_start forecast_origin  target  train_mean
2020-08-03 2020-08-03      2020-08-02   10.80        6.86
2020-08-04 2020-08-03      2020-08-02   21.71        6.86
2020-08-05 2020-08-03      2020-08-02   28.49        6.86
2020-08-06 2020-08-03      2020-08-02   34.36        6.86
2020-08-07 2020-08-03      2020-08-02   11.87        6.86
2020-08-08 2020-08-03      2020-08-02   20.83        6.86
2020-08-09 2020-08-03      2020-08-02    5.75        6.86
2020-08-10 2020-08-10      2020-08-09   16.98       16.39
2020-08-11 2020-08-10      2020-08-09   41.58       16.39
2020-08-12 2020-08-10      2020-08-09   29.75       16.39
2020-08-13 2020-08-10      2020-08-09   25.40       16.39
2020-08-14 2020-08-10      2020-08-09   23.39       16.39
2020-08-15 2020-08-10      2020-08-09   13.39       16.39
2020-08-16 2020-08-10      2020-08-09   12.83       16.39

VAL


## Step 27: Seasonal Naive Forecasting Baseline

We reproduce Stream 2's seven-day seasonal naive benchmark on the Rohlik dataset.

We generate two forecast versions:
- Raw: Based on synthetic availability-censored sales.
- Corrected: Based on Rohlik's internally adjusted sales.

Both versions are evaluated against the same adjusted-sales demand proxy.

We report WAPE, MAE, and forecast bias. We also report the number of evaluated observations to ensure a fair comparison.

For validation and test, forecasts are generated in seven-day blocks using historical sales available at the corresponding forecast origin.


In [167]:

import numpy as np
import pandas as pd

# Create forecasts from the previous week's same weekday
naive_raw = raw_model[
    raw_model["set"].isin(["valid", "test"])
][["unique_id", "date", "set", "adjusted_sales", "availability", "lag7"]].copy()

naive_raw["version"] = "raw"
naive_raw["model"] = "seasonal_naive"
naive_raw["forecast"] = naive_raw["lag7"]

naive_corrected = corrected_model[
    corrected_model["set"].isin(["valid", "test"])
][["unique_id", "date", "set", "adjusted_sales", "availability", "lag7"]].copy()

naive_corrected["version"] = "corrected"
naive_corrected["model"] = "seasonal_naive"
naive_corrected["forecast"] = naive_corrected["lag7"]

naive_forecasts = pd.concat(
    [naive_raw, naive_corrected],
    ignore_index=True
).drop(columns=["lag7"])

def evaluate_forecasts(data):
    results = []

    # Compare on the same observations for both versions
    for split in ["valid", "test"]:
        subset = data[data["set"] == split].copy()

        for version, group in subset.groupby("version"):
            group = group.dropna(
                subset=["adjusted_sales", "forecast"]
            )

            actual = group["adjusted_sales"].to_numpy()
            pred = group["forecast"].to_numpy()

            denom = np.abs(actual).sum()

            results.append({
                "set": split,
                "model": group["model"].iloc[0],
                "version": version,
                "wape": np.abs(actual - pred).sum() / denom,
                "mae": np.abs(actual - pred).mean(),
                "bias": (pred - actual).sum() / denom,
                "n_days": len(group)
            })

    return pd.DataFrame(results)

# Restrict each split to the dates where BOTH versions
# have a valid prediction
naive_forecasts = (
    naive_forecasts.groupby(["set", "unique_id", "date"])
    .filter(lambda x: len(x) == 2 and x["forecast"].notna().all())
)

naive_results = evaluate_forecasts(naive_forecasts)

print("SEASONAL NAIVE RESULTS")
print(naive_results.round(4).to_string(index=False))

print("\nTEST OBSERVATIONS BY VERSION:")
print(
    naive_forecasts[naive_forecasts["set"] == "test"]
    .groupby("version").size()
)

print("\nINTERPRETATION:")
print("Lower WAPE is better.")
print("Negative bias means underforecasting demand.")
print("Positive bias means overforecasting demand.")


SEASONAL NAIVE RESULTS
  set          model   version   wape     mae    bias  n_days
valid seasonal_naive corrected 0.2807 35.2781  0.0312   27739
valid seasonal_naive       raw 0.2866 36.0190 -0.0108   27739
 test seasonal_naive corrected 0.2719 34.3077 -0.0154   14000
 test seasonal_naive       raw 0.2818 35.5592 -0.0505   14000

TEST OBSERVATIONS BY VERSION:
version
corrected    14000
raw          14000
dtype: int64

INTERPRETATION:
Lower WAPE is better.
Negative bias means underforecasting demand.
Positive bias means overforecasting demand.



## Step 28: LightGBM Forecasting

We reproduce Stream 2's LightGBM approach using Rohlik's enriched dataset.

We train two LightGBM models with identical hyperparameters and feature definitions:

1. Raw model: Trained on synthetic availability-censored sales.
2. Corrected model: Trained on Rohlik's internally adjusted sales.

Both models use the same product sample, chronological splits, calendar-aligned historical features, and seven-day forecasting horizon.

We use a Tweedie objective and early stopping on the validation period, following Stream 2's methodology. Each model is validated against its corresponding training target.

Final forecasts will be evaluated against the common adjusted-sales demand proxy.


In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# Match Stream 2's main LightGBM hyperparameters
MEAN_PARAMS_ROHLIK = dict(
    objective="tweedie",
    n_estimators=1500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

rohlik_models = {}
lgbm_forecasts_list = []

for version, data in [
    ("raw", raw_model),
    ("corrected", corrected_model)
]:
    print(f"\nTRAINING LIGHTGBM: {version.upper()}")

    # Historical training data
    train = data[
        (data["set"] == "train") &
        data["lag7"].notna()
    ]

    # Validation data for early stopping
    valid = data[data["set"] == "valid"]

    # Dates to forecast
    predict_rows = data[
        data["set"].isin(["valid", "test"])
    ]

    model = lgb.LGBMRegressor(**MEAN_PARAMS_ROHLIK)

    model.fit(
        train[FEATURES_ROHLIK],
        train["target"],
        eval_set=[
            (valid[FEATURES_ROHLIK], valid["target"])
        ],
        callbacks=[
            lgb.early_stopping(50, verbose=False)
        ]
    )

    rohlik_models[version] = model

    predictions = np.clip(
        model.predict(
            predict_rows[FEATURES_ROHLIK]
        ),
        0,
        None
    )

    output = predict_rows[
        [
            "unique_id",
            "date",
            "set",
            "adjusted_sales",
            "availability"
        ]
    ].copy()

    output["model"] = "lightgbm"
    output["version"] = version
    output["forecast"] = predictions

    lgbm_forecasts_list.append(output)

    print("Training observations:", len(train))
    print("Validation observations:", len(valid))
    print("Best iteration:", model.best_iteration_)
    print("Forecast observations:", len(output))

# Combine both versions
lgbm_forecasts = pd.concat(
    lgbm_forecasts_list,
    ignore_index=True
)

print("\nLIGHTGBM FORECAST SUMMARY:")
print(
    lgbm_forecasts.groupby(
        ["set", "version"]
    )["forecast"].agg(
        ["count", "mean", "min", "max"]
    ).round(2).to_string()
)

print("\nCHECKS:")
print(
    "Missing forecasts:",
    lgbm_forecasts["forecast"].isna().sum()
)
print(
    "Negative forecasts:",
    (lgbm_forecasts["forecast"] < 0).sum()
)



TRAINING LIGHTGBM: RAW
Training observations: 2021303
Validation observations: 28000
Best iteration: 732
Forecast observations: 42000

TRAINING LIGHTGBM: CORRECTED



## Step 29: Forecast Performance Comparison

We compare Seasonal Naive and LightGBM under two data conditions:

1. Raw: Synthetic availability-censored sales.
2. Corrected: Rohlik's internally adjusted sales.

All four approaches are evaluated against the same adjusted-sales demand proxy using identical validation and test observations.

We calculate:
- WAPE: Total absolute forecast error relative to actual demand.
- MAE: Average absolute forecast error.
- Bias: Total overforecasting or underforecasting relative to demand.

Lower WAPE and MAE indicate better forecast accuracy. Bias closer to zero indicates less systematic overforecasting or underforecasting.

These results describe a semi-synthetic experiment, not direct validation against independently observed true demand.


In [ ]:

import numpy as np
import pandas as pd

# Combine all four forecasting approaches
all_rohlik_forecasts = pd.concat(
    [naive_forecasts, lgbm_forecasts],
    ignore_index=True
)

# Keep the same evaluation dates across all four approaches
forecast_counts = (
    all_rohlik_forecasts.groupby(
        ["set", "unique_id", "date"]
    )
    .agg(
        n_forecasts=("forecast", "count"),
        n_models=("model", "nunique"),
        n_versions=("version", "nunique")
    )
    .reset_index()
)

common_dates = forecast_counts[
    (forecast_counts["n_forecasts"] == 4) &
    (forecast_counts["n_models"] == 2) &
    (forecast_counts["n_versions"] == 2)
][["set", "unique_id", "date"]]

comparison_data = all_rohlik_forecasts.merge(
    common_dates,
    on=["set", "unique_id", "date"],
    how="inner"
)

# Evaluate WAPE, MAE and bias
results = []

for (split, model, version), group in comparison_data.groupby(
    ["set", "model", "version"]
):
    actual = group["adjusted_sales"].to_numpy()
    pred = group["forecast"].to_numpy()

    denominator = np.abs(actual).sum()

    results.append({
        "set": split,
        "model": model,
        "version": version,
        "wape": np.abs(actual - pred).sum() / denominator,
        "mae": np.abs(actual - pred).mean(),
        "bias": (pred - actual).sum() / denominator,
        "n_days": len(group)
    })

rohlik_results = pd.DataFrame(results).sort_values(
    ["set", "model", "version"]
)

print("ALL MODEL RESULTS")
print(rohlik_results.round(4).to_string(index=False))

print("\nTEST WAPE COMPARISON (%)")
test_results = rohlik_results[
    rohlik_results["set"] == "test"
]

print(
    (test_results.pivot(
        index="model",
        columns="version",
        values="wape"
    ) * 100).round(2).to_string()
)

print("\nTEST BIAS COMPARISON (%)")
print(
    (test_results.pivot(
        index="model",
        columns="version",
        values="bias"
    ) * 100).round(2).to_string()
)

print("\nEVALUATION OBSERVATIONS")
print(
    rohlik_results.groupby(["set", "model", "version"])[
        "n_days"
    ].first().to_string()
)



## Step 30: Save Forecasting Results

We save the evaluation results and individual forecasts for all four forecasting approaches.

Both Seasonal Naive and LightGBM show lower test WAPE and less negative test bias when trained on retailer-adjusted sales rather than synthetic censored sales.

However, Seasonal Naive outperforms LightGBM on the held-out test week, despite LightGBM performing better during validation.

These findings are based on a semi-synthetic experiment using Rohlik-adjusted sales as the reference demand proxy. They should not be interpreted as direct validation of the original FreshRetailNet demand-recovery model.


In [ ]:

from pathlib import Path
import pandas as pd

# Shared Drive output paths
RESULTS_PATH = PROCESSED_DIR / "results_stream3_v1.csv"
FORECASTS_PATH = PROCESSED_DIR / "forecasts_stream3_v1.parquet"

# Save model evaluation results
rohlik_results.to_csv(RESULTS_PATH, index=False)

# Save forecasts used in the common evaluation
comparison_data.to_parquet(FORECASTS_PATH, index=False)

# Verify both saved files
saved_results = pd.read_csv(RESULTS_PATH)
saved_forecasts = pd.read_parquet(FORECASTS_PATH)

print("FILES SAVED SUCCESSFULLY")

for path in [RESULTS_PATH, FORECASTS_PATH]:
    print(
        path.name,
        "|",
        round(path.stat().st_size / 1024**2, 2),
        "MB"
    )

print("\nVALIDATION:")
print("Saved result rows:", len(saved_results))
print("Saved forecast rows:", len(saved_forecasts))
print("Models:", saved_forecasts["model"].unique().tolist())
print("Versions:", saved_forecasts["version"].unique().tolist())

print("\nSAVED TEST RESULTS:")
print(
    saved_results[saved_results["set"] == "test"]
    .round(4)
    .to_string(index=False)
)



## Step 31: Visualize Forecast Accuracy and Bias

We create presentation-ready charts comparing Seasonal Naive and LightGBM forecasts trained on synthetic censored sales and retailer-adjusted sales.

The first chart compares test WAPE, where lower values indicate better accuracy.

The second chart compares forecast bias. Negative bias indicates systematic underforecasting, while values closer to zero indicate better aggregate calibration.

We save the charts to the shared figures directory for use in our final report and presentation.

These results are based on semi-synthetic censoring, with Rohlik-adjusted sales as the evaluation reference.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Use our saved test results
plot_data = rohlik_results[
    rohlik_results["set"] == "test"
].copy()

models = ["seasonal_naive", "lightgbm"]
model_labels = ["Seasonal Naive", "LightGBM"]
x = np.arange(len(models))
width = 0.34

# Consistent colors across our Stream 3 figures
raw_color = "#D97757"
corrected_color = "#267D9D"

# ---------- FIGURE 1: TEST WAPE ----------
wape_table = plot_data.pivot(
    index="model",
    columns="version",
    values="wape"
).loc[models] * 100

fig, ax = plt.subplots(figsize=(9, 5.5))

bars_raw = ax.bar(
    x - width / 2,
    wape_table["raw"],
    width,
    label="Synthetic raw sales",
    color=raw_color
)

bars_corrected = ax.bar(
    x + width / 2,
    wape_table["corrected"],
    width,
    label="Adjusted sales",
    color=corrected_color
)

ax.bar_label(bars_raw, fmt="%.2f%%", padding=4)
ax.bar_label(bars_corrected, fmt="%.2f%%", padding=4)

ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.set_ylabel("Test WAPE (%)")
ax.set_title(
    "Rohlik: Forecast Accuracy by Training Data",
    fontweight="bold",
    pad=18
)
ax.legend(frameon=False)
ax.set_ylim(0, wape_table.to_numpy().max() * 1.20)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.15)
ax.set_axisbelow(True)

fig.tight_layout()

WAPE_FIG_PATH = FIGURES_DIR / "stream3_test_wape_comparison.png"
fig.savefig(WAPE_FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()


# ---------- FIGURE 2: TEST FORECAST BIAS ----------
bias_table = plot_data.pivot(
    index="model",
    columns="version",
    values="bias"
).loc[models] * 100

fig, ax = plt.subplots(figsize=(9, 5.5))

bars_raw = ax.bar(
    x - width / 2,
    bias_table["raw"],
    width,
    label="Synthetic raw sales",
    color=raw_color
)

bars_corrected = ax.bar(
    x + width / 2,
    bias_table["corrected"],
    width,
    label="Adjusted sales",
    color=corrected_color
)

ax.bar_label(bars_raw, fmt="%.2f%%", padding=4)
ax.bar_label(bars_corrected, fmt="%.2f%%", padding=4)

ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.set_ylabel("Test Forecast Bias (%)")
ax.set_title(
    "Rohlik: Forecast Bias by Training Data",
    fontweight="bold",
    pad=18
)
ax.legend(frameon=False)
ax.set_ylim(bias_table.to_numpy().min() * 1.30, 1)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.15)
ax.set_axisbelow(True)

fig.tight_layout()

BIAS_FIG_PATH = FIGURES_DIR / "stream3_test_bias_comparison.png"
fig.savefig(BIAS_FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("FIGURES SAVED:")
print(WAPE_FIG_PATH.name)
print(BIAS_FIG_PATH.name)
print("\nFiles exist:",
      WAPE_FIG_PATH.exists(),
      BIAS_FIG_PATH.exists())



## Step 32: Forecast Accuracy by Stockout Severity

We examine whether the benefits of using retailer-adjusted sales vary with historical product availability.

We classify products into low, medium, and high partial-availability groups using training-period observations only:

- Low: 0–10% of days with partial availability.
- Medium: More than 10% to 30%.
- High: More than 30%.

For each group, we compare raw and corrected forecasts using the same test observations and calculate WAPE, forecast bias, and the improvement from correction.

This analysis helps determine whether the effects of synthetic stockout censoring are stronger for products with more frequent availability constraints.


In [ ]:

import numpy as np
import pandas as pd

# Historical stockout frequency, calculated using training only
train_availability = (
    rohlik_sample[rohlik_sample["set"] == "train"]
    .groupby("unique_id")["availability"]
    .apply(lambda x: (x < 1).mean())
    .rename("partial_availability_rate")
    .reset_index()
)

train_availability["stockout_group"] = pd.cut(
    train_availability["partial_availability_rate"],
    bins=[-0.001, 0.10, 0.30, 1.0],
    labels=["Low (0-10%)", "Medium (10-30%)", "High (>30%)"]
)

# Attach historical stockout group to test forecasts
stockout_test = comparison_data[
    comparison_data["set"] == "test"
].merge(
    train_availability,
    on="unique_id",
    how="left",
    validate="many_to_one"
)

results_by_stockout = []

for (model, version, group_name), group in stockout_test.groupby(
    ["model", "version", "stockout_group"],
    observed=True
):
    actual = group["adjusted_sales"].to_numpy()
    pred = group["forecast"].to_numpy()

    denominator = np.abs(actual).sum()

    results_by_stockout.append({
        "model": model,
        "version": version,
        "stockout_group": str(group_name),
        "wape": np.abs(actual - pred).sum() / denominator,
        "bias": (pred - actual).sum() / denominator,
        "n_days": len(group),
        "n_products": group["unique_id"].nunique()
    })

stockout_results = pd.DataFrame(results_by_stockout)

print("STOCKOUT SEVERITY RESULTS")
print(
    stockout_results.sort_values(
        ["model", "stockout_group", "version"]
    ).round(4).to_string(index=False)
)

print("\nWAPE IMPROVEMENT FROM CORRECTION (PERCENTAGE POINTS)")
stockout_wape = stockout_results.pivot(
    index=["model", "stockout_group"],
    columns="version",
    values="wape"
)

stockout_wape["improvement_pp"] = (
    stockout_wape["raw"] - stockout_wape["corrected"]
) * 100

print(stockout_wape.round(4).to_string())



## Step 33: Visualize the Effect of Stockout Severity

We visualize the difference in test WAPE between synthetic raw-sales forecasts and retailer-adjusted-sales forecasts across historical partial-availability groups.

Positive improvements indicate that the corrected version achieves lower forecasting error.

The chart highlights whether the benefit of demand correction is consistent across product groups and forecasting methods.

Stockout severity is measured using training-period availability only.


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Order stockout groups from low to high
groups = [
    "Low (0-10%)",
    "Medium (10-30%)",
    "High (>30%)"
]

models = ["seasonal_naive", "lightgbm"]

# Improvement in WAPE percentage points
plot_table = (
    stockout_wape["improvement_pp"]
    .unstack("model")
    .reindex(groups)
)

x = np.arange(len(groups))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5.5))

bars_naive = ax.bar(
    x - width / 2,
    plot_table["seasonal_naive"],
    width,
    label="Seasonal Naive",
    color="#267D9D"
)

bars_lgbm = ax.bar(
    x + width / 2,
    plot_table["lightgbm"],
    width,
    label="LightGBM",
    color="#D97757"
)

ax.axhline(0, color="black", linewidth=1)

ax.bar_label(
    bars_naive,
    fmt="%.2f pp",
    padding=4,
    fontsize=10
)

ax.bar_label(
    bars_lgbm,
    fmt="%.2f pp",
    padding=4,
    fontsize=10
)

ax.set_xticks(x)
ax.set_xticklabels([
    "Low\n(0–10%)",
    "Medium\n(10–30%)",
    "High\n(>30%)"
])

ax.set_ylabel("WAPE Improvement (percentage points)")
ax.set_xlabel("Historical Partial-Availability Frequency")

ax.set_title(
    "Rohlik: Benefit of Corrected Sales by Stockout Severity",
    fontweight="bold",
    pad=18
)

ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.15)
ax.set_axisbelow(True)

# Leave room for labels on positive and negative bars
ax.set_ylim(-0.4, 1.9)

fig.tight_layout()

STOCKOUT_FIG_PATH = (
    FIGURES_DIR / "stream3_stockout_severity.png"
)

fig.savefig(
    STOCKOUT_FIG_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Save the underlying analysis table
STOCKOUT_RESULTS_PATH = (
    PROCESSED_DIR / "results_stream3_by_stockout_v1.csv"
)

stockout_results.to_csv(
    STOCKOUT_RESULTS_PATH,
    index=False
)

print("FILES SAVED:")
print(STOCKOUT_FIG_PATH.name, STOCKOUT_FIG_PATH.exists())
print(STOCKOUT_RESULTS_PATH.name, STOCKOUT_RESULTS_PATH.exists())



## Step 34: Actual vs. Forecasted Demand

We compare aggregate daily forecasts against Rohlik's retailer-adjusted demand proxy during the seven-day test period.

We plot synthetic raw-sales and corrected-sales forecasts separately for Seasonal Naive and LightGBM.

This visualization shows whether the models capture the overall demand pattern and where they systematically overforecast or underforecast.

Daily values are summed across the same 2,000 product series. The adjusted-sales series is an estimated demand reference rather than independently verified true demand.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Use only common test observations
test_fc = comparison_data[
    comparison_data["set"] == "test"
].copy()

# Aggregate forecast totals by day, model and version
daily_forecasts = (
    test_fc.groupby(["date", "model", "version"])["forecast"]
    .sum()
    .reset_index()
)

# Actual demand appears four times per day (one per model/version)
# so count it only once.
daily_actual = (
    test_fc[
        (test_fc["model"] == "seasonal_naive") &
        (test_fc["version"] == "corrected")
    ]
    .groupby("date")["adjusted_sales"]
    .sum()
)

fig, axes = plt.subplots(
    2, 1, figsize=(11, 8), sharex=True
)

model_names = {
    "seasonal_naive": "Seasonal Naive",
    "lightgbm": "LightGBM"
}

for ax, (model_key, model_label) in zip(
    axes, model_names.items()
):
    subset = daily_forecasts[
        daily_forecasts["model"] == model_key
    ]

    ax.plot(
        daily_actual.index,
        daily_actual.values,
        marker="o",
        linewidth=2.5,
        label="Adjusted demand reference",
        color="black"
    )

    for version, color, label in [
        ("raw", "#D97757", "Synthetic raw"),
        ("corrected", "#267D9D", "Corrected")
    ]:
        s = subset[subset["version"] == version]
        ax.plot(
            s["date"],
            s["forecast"],
            marker="o",
            linewidth=2,
            label=label,
            color=color
        )

    ax.set_title(model_label, fontweight="bold")
    ax.set_ylabel("Daily demand (units)")
    ax.grid(alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)

axes[-1].set_xlabel("Test date")

fig.suptitle(
    "Rohlik: Actual vs. Forecasted Demand — Test Week",
    fontsize=15,
    fontweight="bold"
)

fig.autofmt_xdate()
fig.tight_layout()

DAILY_FIG_PATH = (
    FIGURES_DIR / "stream3_actual_vs_forecast.png"
)

fig.savefig(
    DAILY_FIG_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("FILE SAVED:")
print(DAILY_FIG_PATH.name, DAILY_FIG_PATH.exists())



## Step 35: Warehouse-Level Forecasting Performance

We examine forecasting performance across Rohlik's seven warehouses to determine whether the benefits of training on adjusted sales are geographically consistent.

For each warehouse and forecasting model, we compare test WAPE under synthetic raw sales and retailer-adjusted sales.

We also evaluate forecast bias and the number of observations to identify where performance improvements are strongest or weakest.

Warehouse comparisons use the same test dates and selected products for both forecast versions.


In [ ]:

import pandas as pd
import numpy as np

# Add warehouse information to our common test forecasts
warehouse_lookup = (
    rohlik_sample[["unique_id", "warehouse"]]
    .drop_duplicates()
)

warehouse_test = comparison_data[
    comparison_data["set"] == "test"
].merge(
    warehouse_lookup,
    on="unique_id",
    how="left",
    validate="many_to_one"
)

warehouse_rows = []

for (warehouse, model, version), group in warehouse_test.groupby(
    ["warehouse", "model", "version"]
):
    actual = group["adjusted_sales"].to_numpy()
    predicted = group["forecast"].to_numpy()
    denominator = np.abs(actual).sum()

    warehouse_rows.append({
        "warehouse": warehouse,
        "model": model,
        "version": version,
        "wape": np.abs(actual - predicted).sum() / denominator,
        "bias": (predicted - actual).sum() / denominator,
        "n_days": len(group),
        "n_products": group["unique_id"].nunique()
    })

warehouse_results = pd.DataFrame(warehouse_rows)

warehouse_pivot = warehouse_results.pivot(
    index=["warehouse", "model"],
    columns="version",
    values="wape"
)

warehouse_pivot["improvement_pp"] = (
    warehouse_pivot["raw"] -
    warehouse_pivot["corrected"]
) * 100

print("WAREHOUSE-LEVEL WAPE COMPARISON")
print(warehouse_pivot.round(4).to_string())

print("\nWAREHOUSES BENEFITING FROM CORRECTION")
for model in ["seasonal_naive", "lightgbm"]:
    subset = warehouse_pivot.xs(model, level="model")
    print(
        model,
        f"{(subset['improvement_pp'] > 0).sum()} of {len(subset)} warehouses"
    )

print("\nOBSERVATIONS BY WAREHOUSE:")
print(
    warehouse_test[
        (warehouse_test["model"] == "seasonal_naive") &
        (warehouse_test["version"] == "corrected")
    ].groupby("warehouse").agg(
        products=("unique_id", "nunique"),
        observations=("unique_id", "size")
    ).to_string()
)



## Step 36: Warehouse-Level Improvement Visualization

We visualize the test WAPE improvement from using retailer-adjusted sales rather than synthetic censored sales across Rohlik's seven warehouses.

Positive values indicate that adjusted-sales forecasts have lower WAPE. Negative values indicate that synthetic raw-sales forecasts perform better.

This comparison highlights the consistency of the correction benefit and identifies exceptions.

All warehouse comparisons use matched test observations for both forecasting versions.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Use the existing warehouse WAPE improvement table
plot_warehouse = (
    warehouse_pivot["improvement_pp"]
    .unstack("model")
    .sort_index()
)

x = np.arange(len(plot_warehouse))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5.5))

bars_naive = ax.bar(
    x - width / 2,
    plot_warehouse["seasonal_naive"],
    width,
    label="Seasonal Naive",
    color="#267D9D"
)

bars_lgbm = ax.bar(
    x + width / 2,
    plot_warehouse["lightgbm"],
    width,
    label="LightGBM",
    color="#D97757"
)

ax.axhline(0, color="black", linewidth=1)

ax.bar_label(
    bars_naive,
    fmt="%.2f",
    padding=3,
    fontsize=9
)

ax.bar_label(
    bars_lgbm,
    fmt="%.2f",
    padding=3,
    fontsize=9
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_warehouse.index,
    rotation=20,
    ha="right"
)

ax.set_ylabel("Test WAPE Improvement (percentage points)")
ax.set_title(
    "Rohlik: Benefit of Adjusted Sales Across Warehouses",
    fontweight="bold",
    pad=18
)

ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.15)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

ax.set_ylim(-1.1, 3.0)

fig.tight_layout()

WAREHOUSE_FIG_PATH = (
    FIGURES_DIR / "stream3_warehouse_improvement.png"
)

fig.savefig(
    WAREHOUSE_FIG_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Save warehouse-level numerical results
WAREHOUSE_RESULTS_PATH = (
    PROCESSED_DIR / "results_stream3_by_warehouse_v1.csv"
)

warehouse_results.to_csv(
    WAREHOUSE_RESULTS_PATH,
    index=False
)

print("FILES SAVED:")
print(WAREHOUSE_FIG_PATH.name, WAREHOUSE_FIG_PATH.exists())
print(WAREHOUSE_RESULTS_PATH.name, WAREHOUSE_RESULTS_PATH.exists())



## Step 37: Stream 3 Findings and Limitations

### Research Objective

We investigated whether forecasting models trained on stockout-corrected demand outperform models trained on sales affected by product unavailability.

We used the Rohlik Sales Forecasting Challenge V2 dataset as a second-retailer experiment, selecting 2,000 product series across seven warehouses.

### Experimental Design

Rohlik's reported sales are already adjusted for partial product availability using the retailer's internal methodology. We therefore treated reported sales as an estimated demand reference.

To construct a comparable raw-sales condition, we generated semi-synthetic censored sales:

**Synthetic raw sales = Adjusted sales × Availability**

We trained Seasonal Naive and LightGBM models on both versions using matched observations, chronological splits, and seven-day forecasting blocks.

### Main Forecasting Results

On the test set:

| Model | Synthetic Raw WAPE | Corrected WAPE | Improvement |
|---|---:|---:|---:|
| Seasonal Naive | 28.18% | 27.19% | +0.99 pp |
| LightGBM | 28.75% | 28.08% | +0.67 pp |

Corrected sales also reduced aggregate underforecasting:

- Seasonal Naive bias improved from −5.05% to −1.54%.
- LightGBM bias improved from −8.37% to −4.39%.

Seasonal Naive outperformed LightGBM on the held-out test week, despite LightGBM performing better on validation.

### Stockout Severity Findings

The high-stockout group experienced the largest WAPE improvements for both models:

- Seasonal Naive: +1.20 percentage points.
- LightGBM: +1.54 percentage points.

However, improvement was not universal. In the medium-stockout group, corrected LightGBM forecasts performed slightly worse than synthetic raw forecasts.

### Warehouse-Level Findings

Adjusted sales improved Seasonal Naive WAPE in all seven warehouses and LightGBM WAPE in six of seven warehouses.

The largest Seasonal Naive improvement occurred in Prague_3 (+2.47 pp). The largest LightGBM improvement occurred in Munich_1 (+1.40 pp).

LightGBM performance worsened slightly in Prague_2 (−0.72 pp).

### Limitations

1. Rohlik's sales are already internally adjusted. We do not observe independently verified true demand or original uncensored transaction sales.
2. Synthetic raw sales are generated by multiplying adjusted sales by daily availability. This proportional censoring assumption does not necessarily reproduce actual stockout dynamics.
3. Because the retailer-adjusted series is also our evaluation reference, the experiment measures performance relative to that proxy rather than verified demand.
4. The 2,000 selected product series require sufficient historical coverage and complete validation/test dates. The sample may not represent products with highly irregular activity.
5. Results are based on one held-out seven-day test period, so performance may vary across other periods.
6. We did not independently apply or validate Stream 1's FreshRetailNet demand-recovery model on Rohlik.
7. Our adjusted-sales forecast accuracy findings do not directly establish inventory cost savings or financial benefits.

### Conclusion

Our semi-synthetic second-retailer experiment provides supportive evidence that reducing sales censoring can improve demand forecasts, particularly for products with frequent availability constraints.

Both Seasonal Naive and LightGBM achieved lower test WAPE and less aggregate underforecasting when trained on retailer-adjusted sales.

However, improvements varied across product groups and warehouses, and the synthetic design limits claims about real-world demand recovery or transferability of the original correction method.


In [ ]:

from pathlib import Path

FINDINGS_PATH = PROJECT_DIR / "stream3_second_retailer_findings.md"

findings = """
# Stream 3: Second-Retailer Validation — Rohlik

## Methodology

We used 2,000 product series from the Rohlik Sales Forecasting Challenge V2 across seven warehouses.

Because Rohlik sales are already adjusted for availability, we created semi-synthetic censored sales using:

Synthetic raw sales = Adjusted sales × Availability

We trained Seasonal Naive and LightGBM models on both data versions using matched chronological splits and seven-day forecasting blocks.

Both models were evaluated against Rohlik-adjusted sales as a demand proxy.

## Test Results

| Model | Synthetic Raw WAPE | Corrected WAPE | Improvement |
|---|---:|---:|---:|
| Seasonal Naive | 28.18% | 27.19% | +0.99 pp |
| LightGBM | 28.75% | 28.08% | +0.67 pp |

## Major Findings

1. Corrected sales improved test WAPE for both forecasting models.
2. Aggregate underforecasting decreased for both corrected models.
3. Seasonal Naive outperformed LightGBM on the held-out test week.
4. High-stockout products showed the largest WAPE improvements for both models.
5. Seasonal Naive improved in all seven warehouses; LightGBM improved in six of seven.

## Important Exceptions

- LightGBM worsened by approximately 0.10 percentage points in the medium-stockout group.
- LightGBM worsened by approximately 0.72 percentage points in Prague_2.
- Corrected forecasts did not always produce lower absolute bias within every stockout group.

## Limitations

- The experiment uses semi-synthetic censored sales, not independently observed historical transactions.
- Rohlik's adjusted sales are a demand proxy, not verified true demand.
- We did not independently validate Stream 1's demand-recovery model on Rohlik.
- Product eligibility requires complete validation/test coverage.
- We evaluated one held-out seven-day test period.
- Findings do not establish financial benefits or inventory savings.

## Figures

- stream3_test_wape_comparison.png
- stream3_test_bias_comparison.png
- stream3_stockout_severity.png
- stream3_actual_vs_forecast.png
- stream3_warehouse_improvement.png

Figures are stored in Stockout Project Data/figures/.

## Data Source

Rohlik Sales Forecasting Challenge V2:
https://www.kaggle.com/competitions/rohlik-sales-forecasting-challenge-v2

## Conclusion

Our semi-synthetic second-retailer experiment provides supporting evidence that stockout-related sales censoring can reduce demand forecasting accuracy.

Training on retailer-adjusted sales improved overall WAPE and reduced aggregate underforecasting for both Seasonal Naive and LightGBM.

However, results varied across product groups and warehouses, and the experiment does not directly establish the performance of our original missed-demand recovery method on a second retailer.
"""

FINDINGS_PATH.write_text(findings.strip() + "\n", encoding="utf-8")

print("Findings document saved:", FINDINGS_PATH.name)
print("File exists:", FINDINGS_PATH.exists())
print("File size:", FINDINGS_PATH.stat().st_size, "bytes")



## Step 39: Prepare Stream 3 GitHub Contribution

We prepare our completed Stream 3 analysis for integration into the team's GitHub repository.

Our contribution includes the forecasting notebook, evaluation tables, figures, and findings document.

We will work on a separate Git branch and create a pull request for team review.

We will not upload raw Rohlik competition files, Kaggle credentials, or large processed datasets to GitHub.


In [ ]:

import subprocess
from pathlib import Path

REPO_DIR = Path("/content/stockout-aware-forecasting")

print("REPOSITORY EXISTS:", (REPO_DIR / ".git").exists())

commands = [
    ["git", "status", "--short"],
    ["git", "branch", "--show-current"],
    ["git", "remote", "-v"],
    ["git", "log", "-1", "--oneline"]
]

for command in commands:
    print("\n$", " ".join(command))

    result = subprocess.run(
        command,
        cwd=REPO_DIR,
        capture_output=True,
        text=True
    )

    print(result.stdout.strip() or "(no output)")

    if result.returncode != 0:
        print("ERROR:", result.stderr.strip())



## Step 40: Stage Stream 3 Deliverables

We create a dedicated Git branch for the Stream 3 analysis.

We then copy the completed notebook, findings document, evaluation tables, and visualization files into the repository.

We exclude raw Rohlik data, credentials, and large processed datasets.

We verify the copied files before committing or pushing anything.


In [ ]:

import subprocess
import shutil
from pathlib import Path

REPO_DIR = Path("/content/stockout-aware-forecasting")
BRANCH_NAME = "stream3-second-retailer"

# Create our branch from the clean main branch
branch_check = subprocess.run(
    ["git", "branch", "--list", BRANCH_NAME],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    check=True
)

if branch_check.stdout.strip():
    subprocess.run(
        ["git", "switch", BRANCH_NAME],
        cwd=REPO_DIR,
        check=True
    )
else:
    subprocess.run(
        ["git", "switch", "-c", BRANCH_NAME],
        cwd=REPO_DIR,
        check=True
    )

# Find the saved Colab notebook
notebook_matches = list(
    NOTEBOOK_DIR.glob("stream3_second_retailer*.ipynb")
)

print("NOTEBOOK MATCHES:")
for p in notebook_matches:
    print(p.name)

# Copy only when there is one unambiguous match
if len(notebook_matches) != 1:
    raise RuntimeError(
        "Expected exactly one Stream 3 notebook. "
        "Please check the notebook filenames."
    )

# Source files -> repository destinations
file_pairs = [
    (
        notebook_matches[0],
        REPO_DIR / "notebooks" / "stream3_second_retailer.ipynb"
    ),
    (
        FINDINGS_PATH,
        REPO_DIR / "results" / "stream3_second_retailer_findings.md"
    ),
    (
        RESULTS_PATH,
        REPO_DIR / "results" / "tables" / "results_stream3_v1.csv"
    ),
    (
        STOCKOUT_RESULTS_PATH,
        REPO_DIR / "results" / "tables" /
        "results_stream3_by_stockout_v1.csv"
    ),
    (
        WAREHOUSE_RESULTS_PATH,
        REPO_DIR / "results" / "tables" /
        "results_stream3_by_warehouse_v1.csv"
    ),
    (
        WAPE_FIG_PATH,
        REPO_DIR / "figures" /
        "stream3_test_wape_comparison.png"
    ),
    (
        BIAS_FIG_PATH,
        REPO_DIR / "figures" /
        "stream3_test_bias_comparison.png"
    ),
    (
        STOCKOUT_FIG_PATH,
        REPO_DIR / "figures" /
        "stream3_stockout_severity.png"
    ),
    (
        DAILY_FIG_PATH,
        REPO_DIR / "figures" /
        "stream3_actual_vs_forecast.png"
    ),
    (
        WAREHOUSE_FIG_PATH,
        REPO_DIR / "figures" /
        "stream3_warehouse_improvement.png"
    )
]

# Verify all source files before copying any
missing = [str(src) for src, _ in file_pairs if not src.exists()]

if missing:
    raise FileNotFoundError(
        "Missing deliverables:\n" + "\n".join(missing)
    )

for source, destination in file_pairs:
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

print("\nCOPIED DELIVERABLES:")
for _, destination in file_pairs:
    print(
        destination.relative_to(REPO_DIR),
        "|",
        round(destination.stat().st_size / 1024**2, 2),
        "MB"
    )

print("\nCURRENT BRANCH:")
subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR
)

print("\nGIT STATUS:")
subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_DIR
)



## Step 41: Verify Git Branch and Deliverables

We verify that the Stream 3 files were copied to our dedicated Git branch.

Before committing, we check the active branch, changed files, and notebook for potentially exposed credentials.

We will not include raw competition data, access tokens, or other secrets in the GitHub contribution.


In [ ]:

import subprocess
import json
import re

def git_output(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()

print("ACTIVE BRANCH:")
print(git_output("branch", "--show-current") or "(empty)")

print("\nCHANGED FILES:")
print(git_output("status", "--short") or "(no changes)")

print("\nTOTAL COPIED FILES:")
print(sum(destination.exists() for _, destination in file_pairs))

# Inspect the notebook without displaying sensitive values
notebook_path = REPO_DIR / "notebooks" / "stream3_second_retailer.ipynb"

with open(notebook_path, encoding="utf-8") as f:
    notebook_json = json.load(f)

code_text = "\n".join(
    "".join(cell.get("source", []))
    for cell in notebook_json["cells"]
    if cell.get("cell_type") == "code"
)

# Detect common credential assignments without printing secrets
secret_patterns = [
    r'(?i)KAGGLE_KEY\s*=\s*["\'][^"\']+',
    r'(?i)KAGGLE_API_TOKEN\s*=\s*["\'][^"\']+',
    r'(?i)["\']key["\']\s*:\s*["\'][^"\']+',
    r'(?i)github_pat_[A-Za-z0-9_]+',
    r'(?i)ghp_[A-Za-z0-9]+'
]

possible_secrets = [
    i + 1
    for i, pattern in enumerate(secret_patterns)
    if re.search(pattern, code_text)
]

print("\nCREDENTIAL CHECK:")
print("Possible credential patterns found:", len(possible_secrets))
print("Review required:", bool(possible_secrets))

print("\nNOTEBOOK VALIDATION:")
print("Notebook cells:", len(notebook_json["cells"]))
print("Notebook format:", notebook_json.get("nbformat"))



## Step 42: Commit Stream 3 Deliverables

We commit the completed Stream 3 forecasting notebook, results tables, figures, and findings document to our dedicated Git branch.

Only the ten verified Stream 3 deliverables are included.

The commit will prepare our contribution for a pull request without modifying the team's main branch.


In [ ]:

import subprocess

# Confirm the correct branch
branch = git_output("branch", "--show-current")

assert branch == "stream3-second-retailer", (
    f"Unexpected branch: {branch}"
)

# Stage only our 10 Stream 3 deliverables
files_to_commit = [
    str(destination.relative_to(REPO_DIR))
    for _, destination in file_pairs
]

subprocess.run(
    ["git", "add", "--", *files_to_commit],
    cwd=REPO_DIR,
    check=True
)

print("STAGED FILES:")
print(git_output("diff", "--cached", "--name-only"))

print("\nSTAGED FILE COUNT:")
staged_files = git_output(
    "diff", "--cached", "--name-only"
).splitlines()

print(len(staged_files))

assert len(staged_files) == 10, (
    "Expected exactly 10 staged files."
)

# Configure author identity only if missing
author_name = subprocess.run(
    ["git", "config", "user.name"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True
).stdout.strip()

author_email = subprocess.run(
    ["git", "config", "user.email"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True
).stdout.strip()

if not author_name or not author_email:
    print("\nGit author identity is not configured.")
    print("Configure your own name and email before committing.")
else:
    subprocess.run(
        [
            "git", "commit",
            "-m",
            "Add Stream 3 Rohlik second-retailer forecasting analysis"
        ],
        cwd=REPO_DIR,
        check=True
    )

    print("\nLATEST COMMIT:")
    print(git_output("log", "-1", "--oneline"))

    print("\nGIT STATUS:")
    print(git_output("status", "--short") or "(clean)")


In [ ]:

import subprocess

# Set your Git commit identity for this repository only
GIT_NAME = "Yash Shrivastava"
GIT_EMAIL = "YOUR_GITHUB_EMAIL"

subprocess.run(
    ["git", "config", "--local", "user.name", GIT_NAME],
    cwd=REPO_DIR,
    check=True
)

subprocess.run(
    ["git", "config", "--local", "user.email", GIT_EMAIL],
    cwd=REPO_DIR,
    check=True
)

print("Git identity configured successfully.")
print("Name:", GIT_NAME)
print("Email:", GIT_EMAIL)


In [ ]:

# Our 10 files are already staged
subprocess.run(
    [
        "git", "commit",
        "-m",
        "Add Stream 3 Rohlik second-retailer forecasting analysis"
    ],
    cwd=REPO_DIR,
    check=True
)

print("\nLATEST COMMIT:")
print(git_output("log", "-1", "--oneline"))

print("\nGIT STATUS:")
print(git_output("status", "--short") or "(clean)")

print("\nACTIVE BRANCH:")
print(git_output("branch", "--show-current"))


In [ ]:

import subprocess

GIT_NAME = "Yash Shrivastava"
GIT_EMAIL = "REPLACE_WITH_YOUR_GITHUB_EMAIL"

subprocess.run(
    ["git", "config", "--local", "user.name", GIT_NAME],
    cwd=REPO_DIR,
    check=True
)

subprocess.run(
    ["git", "config", "--local", "user.email", GIT_EMAIL],
    cwd=REPO_DIR,
    check=True
)

print("Git identity configured successfully!")
print("Name:", GIT_NAME)


In [ ]:

import subprocess

def check_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        capture_output=True,
        text=True
    )
    print(result.stdout.strip() or "(no output)")
    if result.returncode != 0:
        print("ERROR:", result.stderr.strip())

print("1. CURRENT BRANCH:")
check_git("branch", "--show-current")

print("\n2. GIT STATUS:")
check_git("status", "--short")

print("\n3. LAST COMMIT:")
check_git("log", "-1", "--oneline")

print("\n4. STREAM 3 FILES TRACKED BY GIT:")
check_git("ls-files", "notebooks/stream3_second_retailer.ipynb",
          "results/stream3_second_retailer_findings.md")

print("\n5. FILES PRESENT:")
print(sum(destination.exists() for _, destination in file_pairs), "of 10")


In [ ]:

import subprocess

print("CURRENT BRANCH:")
print(git_output("branch", "--show-current"))

print("\nREMOTE REPOSITORY:")
print(git_output("remote", "get-url", "origin"))

print("\nCOMMIT READY TO PUSH:")
print(git_output("log", "-1", "--oneline"))

print("\nUPSTREAM BRANCH:")
result = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "@{upstream}"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout.strip())
else:
    print("No upstream configured yet (expected).")

print("\nGITHUB CLI AUTHENTICATION:")
gh = subprocess.run(
    ["gh", "auth", "status"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True
)

print("Authenticated:", gh.returncode == 0)


In [ ]:

import subprocess

# Your personal GitHub fork
FORK_URL = "https://github.com/Bboy5555/stockout-aware-forecasting.git"

# Keep the team's repository as 'origin'
# Add your fork separately as 'myfork'
remotes = subprocess.run(
    ["git", "remote"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    check=True
).stdout.splitlines()

if "myfork" in remotes:
    subprocess.run(
        ["git", "remote", "set-url", "myfork", FORK_URL],
        cwd=REPO_DIR,
        check=True
    )
else:
    subprocess.run(
        ["git", "remote", "add", "myfork", FORK_URL],
        cwd=REPO_DIR,
        check=True
    )

print("GIT REMOTES:")
print(git_output("remote", "-v"))

print("\nACTIVE BRANCH:")
print(git_output("branch", "--show-current"))

print("\nCOMMIT:")
print(git_output("log", "-1", "--oneline"))

print("\nWORKING TREE:")
print(git_output("status", "--short") or "(clean)")


In [ ]:

import subprocess
import shutil

# Check whether GitHub CLI is installed
print("GitHub CLI installed:", shutil.which("gh") is not None)

if shutil.which("gh"):
    result = subprocess.run(
        ["gh", "auth", "status"],
        capture_output=True,
        text=True
    )

    print("Already authenticated:", result.returncode == 0)
else:
    print("GitHub CLI needs to be installed.")


In [ ]:

!gh auth status


In [ ]:

import subprocess

# Start GitHub login without interactive questions
process = subprocess.Popen(
    ["gh", "auth", "login",
     "--hostname", "github.com",
     "--git-protocol", "https",
     "--web"],
    stdin=subprocess.PIPE,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    text=True
)

process.stdin.write("Y\n")
process.stdin.flush()

print("Login process started.")
print("If no verification code appears, we will use another method.")


In [ ]:
!yes | gh auth login --hostname github.com --git-protocol https --web

In [ ]:

import subprocess

result = subprocess.run(
    ["gh", "auth", "status"],
    capture_output=True,
    text=True
)

print("GitHub authenticated:", result.returncode == 0)

if result.returncode == 0:
    user = subprocess.run(
        ["gh", "api", "user", "--jq", ".login"],
        capture_output=True,
        text=True
    )
    print("GitHub username:", user.stdout.strip())
else:
    print("Login has not finished in Colab.")


In [ ]:

import subprocess

# Allow Git to use the authenticated GitHub CLI
subprocess.run(
    ["gh", "auth", "setup-git"],
    check=True
)

# Verify branch before pushing
branch = git_output("branch", "--show-current")
assert branch == "stream3-second-retailer", f"Unexpected branch: {branch}"

# Push only to your personal fork, not the team repository
subprocess.run(
    ["git", "push", "-u", "myfork", "stream3-second-retailer"],
    cwd=REPO_DIR,
    check=True
)

print("\nPUSH COMPLETED")
print("Branch:", branch)
print("Commit:", git_output("log", "-1", "--oneline"))

print("\nREMOTE BRANCH:")
print(git_output("branch", "-vv"))



## Step 58: FreshRetailNet vs. Rohlik Comparison

We compare the original FreshRetailNet dataset with Rohlik to evaluate how the two retailers differ in historical coverage, product availability, and forecasting performance.

We distinguish FreshRetailNet's observed stockout information from Rohlik's daily availability fractions.

We also distinguish FreshRetailNet's estimated recovered demand from Rohlik's internally adjusted sales and our semi-synthetic censoring experiment.

These differences are important when interpreting the generalizability of our findings.


In [ ]:

import pandas as pd

# Inspect the team's original FreshRetailNet datasets
frn_paths = {
    "Daily model": PROCESSED_DIR / "daily_model.parquet",
    "Corrected demand": PROCESSED_DIR / "daily_demand_v1.parquet"
}

for name, path in frn_paths.items():
    print("\n" + "=" * 45)
    print(name)
    print("File exists:", path.exists())

    if path.exists():
        data_check = pd.read_parquet(path)

        print("Rows:", len(data_check))
        print("Columns:", data_check.columns.tolist())

        for col in ["dt", "date"]:
            if col in data_check.columns:
                print("Date range:",
                      data_check[col].min(),
                      "to",
                      data_check[col].max())
                break

        print("\nFIRST 3 ROWS:")
        print(data_check.head(3).to_string(index=False))

print("\nROHLIK REFERENCE:")
print("Selected products:", rohlik_sample["unique_id"].nunique())
print("Selected rows:", len(rohlik_sample))
print("Historical partial-availability rate: 24.56%")
print("Test Seasonal Naive WAPE: raw 28.18%, corrected 27.19%")
print("Test LightGBM WAPE: raw 28.75%, corrected 28.08%")
